# LeetCode 4114 — Count Consistent Subarrays II (Hard)

Notebook này dạy từ căn bản bài toán:

```python
class Solution:
    def countConsistent(self, nums: List[int]) -> int:
        ...
```

Đây là một bài **Hard** kết hợp ba ý tưởng rất hay gặp: **phép toán bit AND**, **kỹ thuật đếm theo đóng góp (contribution technique)** và **ngăn xếp đơn điệu (monotonic stack)**. Đừng lo nếu bạn chưa biết thuật ngữ nào trong số đó: notebook giải thích từng thứ từ con số 0, có ví dụ tính tay và cell code chạy thử.

---

## Sau notebook này, bạn sẽ nắm được

1. Đề bài yêu cầu gì: "mảng con consistent" là gì, kiểm tra bằng tay một mảng con ra sao.
2. Cách đọc số ở dạng nhị phân và cách phép `&` (AND bit) hoạt động trên từng cột bit.
3. Vì sao **AND của một mảng con không bao giờ lớn hơn min** của nó, và chính xác khi nào hai giá trị này bằng nhau.
4. Mẹo "**cố định đầu phải thì AND chỉ có ≤ 20 giá trị khác nhau**" — kỹ thuật chung cho các bài AND/OR/GCD trên mảng con.
5. Kỹ thuật **đóng góp**: thay vì liệt kê từng mảng con, hỏi "mỗi phần tử đại diện cho bao nhiêu mảng con?", và cách **phá hòa** (tie-breaking) khi có phần tử trùng nhau.
6. Cách dùng **monotonic stack** với một quan hệ khác dấu `<=` quen thuộc (quan hệ "chứa mọi bit"), và vì sao nó vẫn đúng (tính bắc cầu).
7. Cách viết lời giải `O(n)` đúng format LeetCode, tự kiểm thử bằng brute force và tránh các lỗi đếm trùng / đếm thiếu.

## Lộ trình các cách giải

| Cách | Ý tưởng chính | Thời gian |
|---|---|---|
| Cách 1 | Brute force: thử mọi mảng con, tính AND và min từ đầu | `O(n³)` |
| Cách 2 | Cố định đầu trái, mở rộng đầu phải, cập nhật AND/min dần dần | `O(n²)` |
| Cách 3 | Cố định đầu phải: AND chỉ có ≤ 20 giá trị khác nhau + nhớ vị trí xuất hiện cuối | `O(n · B)`, `B ≤ 20` |
| Cách 4 | Đóng góp của từng phần tử + monotonic stack theo quan hệ "chứa mọi bit" | `O(n)` |

Ta đi từ cách dễ nghĩ nhất lên cách tốt nhất; mỗi bước đều chỉ rõ vì sao cần cải tiến.

# 1. Đề bài

Cho một mảng số nguyên `nums`.

Một **mảng con** (subarray) là một đoạn **liên tiếp** của mảng. Ký hiệu `nums[l..r]` nghĩa là các phần tử từ chỉ số `l` đến chỉ số `r` (tính cả hai đầu).

Một mảng con được gọi là **consistent** (nhất quán) nếu:

> **AND bit của tất cả phần tử** trong mảng con **bằng giá trị nhỏ nhất** (min) của mảng con đó.

Hãy trả về **số lượng mảng con consistent** của `nums`.

## Ví dụ 1

```text
Input:  nums = [5, 3, 1, 2]
Output: 6
```

Các mảng con consistent:

| Mảng con | Giá trị | AND bit | Min |
|---|---|---|---:|
| `nums[0..0]` | `[5]` | 5 | 5 |
| `nums[1..1]` | `[3]` | 3 | 3 |
| `nums[2..2]` | `[1]` | 1 | 1 |
| `nums[3..3]` | `[2]` | 2 | 2 |
| `nums[1..2]` | `[3, 1]` | 3 AND 1 = 1 | 1 |
| `nums[0..2]` | `[5, 3, 1]` | 5 AND 3 AND 1 = 1 | 1 |

Vậy có 6 mảng con consistent.

## Ví dụ 2

```text
Input:  nums = [1, 1, 1]
Output: 6
```

Mọi mảng con đều có AND bằng 1 và min bằng 1, nên cả 6 mảng con đều consistent.

## Ví dụ 3

```text
Input:  nums = [0]
Output: 1
```

Mảng con duy nhất là `[0]`, AND và min đều bằng 0.

## Ràng buộc (constraints)

```text
1 <= nums.length <= 2 * 10^5
0 <= nums[i] <= 10^6
```

## 1.1 Ràng buộc nói gì về độ phức tạp cần đạt?

Gọi `n = len(nums)`.

- Số mảng con của một mảng dài `n` là `n·(n+1)/2`. Với `n = 2·10^5`, con số này khoảng **2·10^10**. Chỉ riêng việc "chạm" vào mỗi mảng con một lần đã quá chậm (Python làm được cỡ `10^7` phép tính đơn giản mỗi giây).
- Vậy ta **không thể liệt kê từng mảng con**. Cần thuật toán cỡ `O(n)`, `O(n log n)` hoặc `O(n · 20)`.
- `nums[i] <= 10^6 < 2^20 = 1 048 576`, nên mỗi số có **tối đa 20 bit**. Con số nhỏ "20" là một gợi ý mạnh: mẹo theo bit chỉ tốn thêm hệ số khoảng 20, vẫn chấp nhận được.
- Đáp án có thể lớn tới `n·(n+1)/2 ≈ 2·10^10`, vượt giới hạn số nguyên 32-bit (`2 147 483 647`). Python tự xử lý số lớn, nhưng nếu viết Java/C++ thì phải dùng `long` / `long long`.

Cell dưới in ra dữ liệu ví dụ và vài con số vừa nhắc.

In [1]:
from typing import List

examples = [
    ([5, 3, 1, 2], 6),
    ([1, 1, 1], 6),
    ([0], 1),
]

for nums, expected in examples:
    n = len(nums)
    print(f"nums = {nums}, n = {n}, số mảng con = {n * (n + 1) // 2}, đáp án mong đợi = {expected}")

n_max = 2 * 10**5
print()
print("Số mảng con khi n = 2*10^5:", n_max * (n_max + 1) // 2)
print("2^20 =", 2**20, "| 2^20 > 10^6 ?", 2**20 > 10**6)
print("Số bit cần để viết 10^6:", (10**6).bit_length())
print("Giới hạn int 32-bit:", 2**31 - 1)

nums = [5, 3, 1, 2], n = 4, số mảng con = 10, đáp án mong đợi = 6
nums = [1, 1, 1], n = 3, số mảng con = 6, đáp án mong đợi = 6
nums = [0], n = 1, số mảng con = 1, đáp án mong đợi = 1

Số mảng con khi n = 2*10^5: 20000100000
2^20 = 1048576 | 2^20 > 10^6 ? True
Số bit cần để viết 10^6: 20
Giới hạn int 32-bit: 2147483647


# 2. Hiểu đề bằng tay trước khi nghĩ thuật toán

## 2.1 Liệt kê toàn bộ 10 mảng con của `[5, 3, 1, 2]`

Viết các số ở dạng nhị phân (3 bit) để tính AND theo từng cột:

```text
5 = 101
3 = 011
1 = 001
2 = 010
```

AND theo cột: một cột cho ra `1` **chỉ khi tất cả** các số đều có `1` ở cột đó.

```text
[5, 3]      : 101 & 011       = 001 = 1 , min = 3  -> 1 != 3 -> KHÔNG
[5, 3, 1]   : 101 & 011 & 001 = 001 = 1 , min = 1  -> 1 == 1 -> CÓ
[3, 1, 2]   : 011 & 001 & 010 = 000 = 0 , min = 1  -> 0 != 1 -> KHÔNG
```

Bảng đầy đủ:

| l | r | Mảng con | AND | min | Consistent? |
|---:|---:|---|---:|---:|:---:|
| 0 | 0 | `[5]` | 5 | 5 | Có |
| 0 | 1 | `[5, 3]` | 1 | 3 | Không |
| 0 | 2 | `[5, 3, 1]` | 1 | 1 | Có |
| 0 | 3 | `[5, 3, 1, 2]` | 0 | 1 | Không |
| 1 | 1 | `[3]` | 3 | 3 | Có |
| 1 | 2 | `[3, 1]` | 1 | 1 | Có |
| 1 | 3 | `[3, 1, 2]` | 0 | 1 | Không |
| 2 | 2 | `[1]` | 1 | 1 | Có |
| 2 | 3 | `[1, 2]` | 0 | 1 | Không |
| 3 | 3 | `[2]` | 2 | 2 | Có |

Đếm số dòng "Có": **6**, khớp với đề bài.

## 2.2 Ba quan sát nhỏ (sẽ được chứng minh ở phần "Bước đột phá")

1. Ở **mọi dòng**, cột AND **không bao giờ lớn hơn** cột min.
2. Mảng con một phần tử luôn consistent (AND của một số là chính nó, min cũng là nó).
3. `[5, 3]` **không** consistent nhưng kéo dài thành `[5, 3, 1]` thì **lại** consistent: thêm phần tử có thể "cứu" một mảng con. Nghĩa là khi duyệt, ta **không được dừng sớm** chỉ vì gặp một mảng con xấu.

## 2.3 Một ví dụ tự đặt: `[2, 1, 0]`

| Mảng con | AND | min | Consistent? |
|---|---:|---:|:---:|
| `[2]` | 2 | 2 | Có |
| `[2, 1]` | 0 | 1 | Không |
| `[2, 1, 0]` | 0 | 0 | Có |
| `[1]` | 1 | 1 | Có |
| `[1, 0]` | 0 | 0 | Có |
| `[0]` | 0 | 0 | Có |

Đáp án: **5**. Để ý: hễ mảng con chứa số `0` thì min = 0 và AND = 0, nên luôn consistent.

Ta sẽ dùng `[2, 1, 0]` như một ví dụ "bẫy" xuyên suốt notebook.

# 3. Nền tảng cần biết

Trước khi giải, ta học (hoặc ôn) từng công cụ sẽ dùng. Mỗi mục đi theo nhịp: **nó là gì → ví dụ nhỏ → code chạy thử → liên quan gì tới bài**.

| Mục | Công cụ | Dùng ở đâu |
|---|---|---|
| 3.1 | Mảng con, slicing `nums[l:r+1]` | Cách 1, Cách 2 |
| 3.2 | Biểu diễn nhị phân, `bin`, `format`, `>>` | Hiểu phép AND |
| 3.3 | Phép AND bit `&`, `&=` | Mọi cách |
| 3.4 | Quan hệ "chứa mọi bit": `(a & b) == b` | Bước đột phá, Cách 4 |
| 3.5 | `dict` nhớ vị trí xuất hiện cuối, `enumerate`, `.get` | Cách 3 |
| 3.6 | Stack bằng `list`, monotonic stack | Cách 4 |
| 3.7 | Kỹ thuật đóng góp `(i - L) * (R - i)` | Cách 4 |
| 3.8 | Phá hòa khi có phần tử trùng | Cách 4 |

## 3.1 Mảng con và slicing

Trong Python, `nums[l:r + 1]` lấy các phần tử từ chỉ số `l` đến chỉ số `r`. Đầu cuối `r + 1` **không** được lấy, nên phải cộng 1 thì mới lấy được `nums[r]`.

Mỗi mảng con ứng với đúng một cặp `(l, r)` với `0 <= l <= r <= n - 1`. Với `l = 0` có `n` lựa chọn của `r`, với `l = 1` có `n - 1` lựa chọn, ... nên tổng số mảng con là:

```text
n + (n - 1) + ... + 1 = n·(n+1)/2
```

In [2]:
nums = [5, 3, 1, 2]
n = len(nums)

count = 0
for l in range(n):
    for r in range(l, n):
        sub = nums[l:r + 1]          # lấy từ chỉ số l đến r (tính cả r)
        print(f"l={l}, r={r}, sub={sub}, min={min(sub)}")
        count += 1

print("Tổng số mảng con:", count, "| công thức n(n+1)/2 =", n * (n + 1) // 2)

l=0, r=0, sub=[5], min=5
l=0, r=1, sub=[5, 3], min=3
l=0, r=2, sub=[5, 3, 1], min=1
l=0, r=3, sub=[5, 3, 1, 2], min=1
l=1, r=1, sub=[3], min=3
l=1, r=2, sub=[3, 1], min=1
l=1, r=3, sub=[3, 1, 2], min=1
l=2, r=2, sub=[1], min=1
l=2, r=3, sub=[1, 2], min=1
l=3, r=3, sub=[2], min=2
Tổng số mảng con: 10 | công thức n(n+1)/2 = 10


## 3.2 Biểu diễn nhị phân

Mỗi số nguyên không âm viết được thành tổng các lũy thừa của 2:

```text
5 = 4 + 1     = 1·2² + 0·2¹ + 1·2⁰ = 101 (nhị phân)
6 = 4 + 2     = 110
3 = 2 + 1     = 011
```

**Bit thứ `k`** (đếm từ phải sang, bắt đầu từ 0) có giá trị `2^k`. Ví dụ `6 = 110`: bit 0 là 0, bit 1 là 1, bit 2 là 1.

Công cụ Python:

- `bin(5)` → chuỗi `'0b101'` (tiền tố `0b` nghĩa là "binary").
- `format(5, '03b')` → `'101'`: in nhị phân, đủ 3 chữ số (thêm `0` bên trái nếu thiếu).
- `(x >> k) & 1` → lấy bit thứ `k` của `x` (dịch phải `k` bước rồi lấy bit cuối).
- `x.bit_length()` → số bit cần để viết `x`.

In [3]:
for x in [5, 3, 1, 2, 7, 6, 0]:
    print(f"{x} -> bin = {bin(x):>6} -> 3 bit: {format(x, '03b')}")

x = 6
for k in range(3):
    print(f"bit thứ {k} của {x} là {(x >> k) & 1}")

5 -> bin =  0b101 -> 3 bit: 101
3 -> bin =   0b11 -> 3 bit: 011
1 -> bin =    0b1 -> 3 bit: 001
2 -> bin =   0b10 -> 3 bit: 010
7 -> bin =  0b111 -> 3 bit: 111
6 -> bin =  0b110 -> 3 bit: 110
0 -> bin =    0b0 -> 3 bit: 000
bit thứ 0 của 6 là 0
bit thứ 1 của 6 là 1
bit thứ 2 của 6 là 1


## 3.3 Phép AND bit `&`

Bảng chân trị (truth table) cho một bit:

| a | b | a & b |
|:-:|:-:|:-:|
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

Với hai số, `&` làm việc **trên từng cột bit một cách độc lập**:

```text
  5 = 101
  3 = 011
5&3 = 001 = 1
```

AND của nhiều số: bắt đầu bằng phần tử đầu, rồi lần lượt `result &= v` (viết tắt của `result = result & v`).

Hai tính chất sẽ dùng rất nhiều lần:

- **(P1)** `a & b` chỉ giữ những bit mà **cả hai** đều có, nên kết quả "nhỏ hơn hoặc bằng" từng số: `a & b <= a` và `a & b <= b`.
- **(P2)** Khi thêm một phần tử vào mảng con, AND chỉ có thể **giữ nguyên hoặc mất bớt bit**, không bao giờ được thêm bit. Nói cách khác: kéo dài mảng con thì AND không tăng.

In [4]:
from functools import reduce
import operator

print("5 & 3 =", 5 & 3, "->", format(5, '03b'), "&", format(3, '03b'), "=", format(5 & 3, '03b'))
print("6 & 3 =", 6 & 3)
print("7 & 5 =", 7 & 5)


def and_of(values):
    result = values[0]
    for v in values[1:]:
        result &= v          # result = result & v
    return result


print("AND của [5, 3, 1] =", and_of([5, 3, 1]))
print("Dùng reduce:", reduce(operator.and_, [5, 3, 1]))

# (P2): kéo dài dần mảng con [7, 6, 5, 4, 1] -> AND chỉ mất bit, không tăng
running = None
for v in [7, 6, 5, 4, 1]:
    running = v if running is None else running & v
    print(f"thêm {v}: AND = {running} ({format(running, '03b')})")

5 & 3 = 1 -> 101 & 011 = 001
6 & 3 = 2
7 & 5 = 5
AND của [5, 3, 1] = 1
Dùng reduce: 1
thêm 7: AND = 7 (111)
thêm 6: AND = 6 (110)
thêm 5: AND = 4 (100)
thêm 4: AND = 4 (100)
thêm 1: AND = 0 (000)


Giờ đã có `&`, ta cho máy tính lập lại **bảng ở mục 2.1** và kiểm tra ví dụ `[2, 1, 0]` ở mục 2.3.

Một chi tiết Python: `ok = (a == m)` là kiểu `bool`. Khi cộng, `True` được tính là `1` và `False` là `0`, nên `good += ok` đếm được số dòng "Có".

In [5]:
def show_table(nums):
    n = len(nums)
    good = 0
    for l in range(n):
        for r in range(l, n):
            sub = nums[l:r + 1]
            a = and_of(sub)
            m = min(sub)
            ok = (a == m)
            good += ok                      # True -> +1, False -> +0
            print(f"l={l} r={r} sub={str(sub):<14} AND={a:<2} min={m:<2} {'Có' if ok else 'Không'}")
    print("Số mảng con consistent:", good)


show_table([5, 3, 1, 2])
print()
show_table([2, 1, 0])

l=0 r=0 sub=[5]            AND=5  min=5  Có
l=0 r=1 sub=[5, 3]         AND=1  min=3  Không
l=0 r=2 sub=[5, 3, 1]      AND=1  min=1  Có
l=0 r=3 sub=[5, 3, 1, 2]   AND=0  min=1  Không
l=1 r=1 sub=[3]            AND=3  min=3  Có
l=1 r=2 sub=[3, 1]         AND=1  min=1  Có
l=1 r=3 sub=[3, 1, 2]      AND=0  min=1  Không
l=2 r=2 sub=[1]            AND=1  min=1  Có
l=2 r=3 sub=[1, 2]         AND=0  min=1  Không
l=3 r=3 sub=[2]            AND=2  min=2  Có
Số mảng con consistent: 6

l=0 r=0 sub=[2]            AND=2  min=2  Có
l=0 r=1 sub=[2, 1]         AND=0  min=1  Không
l=0 r=2 sub=[2, 1, 0]      AND=0  min=0  Có
l=1 r=1 sub=[1]            AND=1  min=1  Có
l=1 r=2 sub=[1, 0]         AND=0  min=0  Có
l=2 r=2 sub=[0]            AND=0  min=0  Có
Số mảng con consistent: 5


## 3.4 Quan hệ "chứa mọi bit" (submask / supermask)

Ta nói **`a` chứa mọi bit của `b`** khi mọi bit `1` của `b` cũng là bit `1` của `a`. Thuật ngữ: `b` là **submask** của `a`, `a` là **supermask** của `b`. Trong notebook, ta viết tắt là `b ⊆ a` hoặc `a ⊇ b`.

Cách kiểm tra bằng code:

```python
(a & b) == b
```

Vì sao? `a & b` giữ lại những bit của `b` mà `a` cũng có. Nếu kết quả vẫn bằng nguyên `b`, nghĩa là `a` có đủ mọi bit của `b`.

Ví dụ:

- `7 = 111` chứa `3 = 011`: `7 & 3 = 3` → đúng.
- `5 = 101` **không** chứa `2 = 010`: `5 & 2 = 0 != 2` → sai, **dù 5 > 2**.
- `0` là submask của mọi số: `a & 0 = 0`.
- Mọi số đều chứa chính nó: `a & a = a`.

Ba sự thật quan trọng:

- **(F1)** Nếu `a ⊇ b` thì `a >= b` (vì `a` = `b` + vài bit thêm). Chiều ngược lại **SAI**: `5 >= 2` nhưng `5` không chứa `2`.
- **(F2)** **Bắc cầu (transitive)**: `a ⊇ b` và `b ⊇ c` thì `a ⊇ c`. (Bit của `c` nằm trong `b`, bit của `b` nằm trong `a`.)
- **(F3)** Bản "thật sự" (strict) cũng bắc cầu: nếu `a ⊇ b, a != b` và `b ⊇ c, b != c` thì `a ⊇ c, a != c` (vì khi đó `a > b > c`).

**Lưu ý thứ tự ưu tiên toán tử**: trong Python, `&` được tính **trước** `==`, nên `a & b == b` vẫn hiểu là `(a & b) == b`. Nhưng trong C++/Java/JavaScript thì ngược lại! Hãy luôn viết ngoặc `(a & b) == b` cho chắc.

In [6]:
def contains(a: int, b: int) -> bool:
    # True nếu a chứa mọi bit 1 của b (b là submask của a)
    return (a & b) == b


pairs = [(7, 3), (5, 2), (6, 2), (3, 3), (9, 0), (2, 5)]
for a, b in pairs:
    print(f"a={a} ({format(a, '04b')}), b={b} ({format(b, '04b')}): "
          f"contains={contains(a, b)!s:<5} | a >= b: {a >= b}")

a=7 (0111), b=3 (0011): contains=True  | a >= b: True
a=5 (0101), b=2 (0010): contains=False | a >= b: True
a=6 (0110), b=2 (0010): contains=True  | a >= b: True
a=3 (0011), b=3 (0011): contains=True  | a >= b: True
a=9 (1001), b=0 (0000): contains=True  | a >= b: True
a=2 (0010), b=5 (0101): contains=False | a >= b: False


In [7]:
# Kiểm chứng (F1) và (F2) cho mọi bộ số từ 0 đến 31
ok_f1 = all(a >= b for a in range(32) for b in range(32) if contains(a, b))
ok_f2 = all(contains(a, c)
            for a in range(32) for b in range(32) for c in range(32)
            if contains(a, b) and contains(b, c))
counter = [(a, b) for a in range(32) for b in range(32) if a >= b and not contains(a, b)]

print("F1 (chứa mọi bit => lớn hơn hoặc bằng):", ok_f1)
print("F2 (bắc cầu):", ok_f2)
print("Số cặp a >= b nhưng a KHÔNG chứa b:", len(counter), "| vài ví dụ:", counter[:5])

F1 (chứa mọi bit => lớn hơn hoặc bằng): True
F2 (bắc cầu): True
Số cặp a >= b nhưng a KHÔNG chứa b: 285 | vài ví dụ: [(2, 1), (4, 1), (4, 2), (4, 3), (5, 2)]


Kết luận của mục 3.4: **"lớn hơn" và "chứa mọi bit" là hai quan hệ khác nhau.** "Chứa mọi bit" mạnh hơn: nó kéo theo "lớn hơn hoặc bằng", nhưng không ngược lại. Bài này xoay quanh đúng sự khác biệt đó.

## 3.5 `dict`: nhớ vị trí xuất hiện cuối cùng của mỗi giá trị

- `enumerate(nums)` cho từng cặp `(i, x)`: chỉ số và giá trị.
- `last = {}` là một dictionary (bảng tra `key -> value`).
- `last[x] = i` **ghi đè** giá trị cũ, nên sau khi duyệt tới `i`, `last[x]` luôn là **lần xuất hiện mới nhất** của `x`.
- `last.get(key, -1)` trả về `-1` nếu `key` chưa có (thay vì báo lỗi `KeyError` như `last[key]`).

Cách 3 sẽ dùng đúng bảng `last` này để hỏi: "giá trị `a` có xuất hiện trong đoạn `[l..r]` không?"

In [8]:
nums = [1, 3, 5, 1, 2]
last = {}
for i, x in enumerate(nums):
    last[x] = i                        # ghi đè => luôn là lần xuất hiện mới nhất
    print(f"i={i}, x={x}, last={last}")

print("last.get(1, -1) =", last.get(1, -1))
print("last.get(4, -1) =", last.get(4, -1), "(4 chưa từng xuất hiện)")

i=0, x=1, last={1: 0}
i=1, x=3, last={1: 0, 3: 1}
i=2, x=5, last={1: 0, 3: 1, 5: 2}
i=3, x=1, last={1: 3, 3: 1, 5: 2}
i=4, x=2, last={1: 3, 3: 1, 5: 2, 2: 4}
last.get(1, -1) = 3
last.get(4, -1) = -1 (4 chưa từng xuất hiện)


## 3.6 Stack và monotonic stack

**Stack** (ngăn xếp) là cấu trúc "vào sau ra trước" (LIFO — Last In, First Out), giống chồng đĩa: chỉ lấy được đĩa trên cùng. Trong Python dùng `list`:

- `stack.append(v)`: đặt `v` lên đỉnh (push).
- `stack.pop()`: lấy phần tử ở đỉnh ra (pop).
- `stack[-1]`: xem đỉnh mà không lấy ra (peek).
- `while stack and ...`: luôn kiểm tra stack **khác rỗng trước** rồi mới đọc `stack[-1]`.

In [9]:
stack = []
stack.append(10)
stack.append(20)
stack.append(30)
print("stack:", stack, "| đỉnh:", stack[-1])
print("pop ->", stack.pop(), "| còn lại:", stack)
print("stack rỗng?", not stack)

stack: [10, 20, 30] | đỉnh: 30
pop -> 30 | còn lại: [10, 20]
stack rỗng? False


### Bài toán kinh điển: "phần tử nhỏ hơn gần nhất bên trái"

Với mỗi `i`, tìm chỉ số `j < i` **gần nhất** sao cho `nums[j] < nums[i]` (không có thì `-1`).

- Cách ngây thơ: từ `i` đi lùi từng bước → `O(n²)` trong trường hợp xấu.
- Cách stack: duyệt trái → phải, stack giữ các **chỉ số "ứng viên"**. Khi gặp `x = nums[i]`, ta **pop mọi đỉnh có giá trị `>= x`**: chúng không bao giờ là đáp án cho ai phía sau nữa, vì `x` vừa đứng gần hơn vừa không lớn hơn. Sau khi pop, đỉnh còn lại (nếu có) chính là đáp án của `i`. Cuối cùng đẩy `i` vào.

Mỗi chỉ số được push đúng 1 lần và pop tối đa 1 lần, nên tổng số thao tác `<= 2n`: **O(n) khấu hao** (amortized — tức là tính gộp cả quá trình, dù một bước riêng lẻ có thể pop nhiều lần).

Gọi là **monotonic** (đơn điệu) vì các giá trị trong stack luôn tăng dần từ đáy lên đỉnh.

In [10]:
def previous_smaller(nums):
    # left[i] = chỉ số j < i gần nhất có nums[j] < nums[i], hoặc -1
    left = [-1] * len(nums)
    stack = []                      # chứa CHỈ SỐ, không phải giá trị
    for i, x in enumerate(nums):
        while stack and nums[stack[-1]] >= x:
            stack.pop()             # không bao giờ là đáp án cho ai phía sau nữa
        if stack:
            left[i] = stack[-1]
        stack.append(i)
        print(f"i={i}, x={x}, stack (giá trị) = {[nums[j] for j in stack]}, left[{i}] = {left[i]}")
    return left


print(previous_smaller([5, 7, 2, 3, 2, 6]))

i=0, x=5, stack (giá trị) = [5], left[0] = -1
i=1, x=7, stack (giá trị) = [5, 7], left[1] = 0
i=2, x=2, stack (giá trị) = [2], left[2] = -1
i=3, x=3, stack (giá trị) = [2, 3], left[3] = 2
i=4, x=2, stack (giá trị) = [2], left[4] = -1
i=5, x=6, stack (giá trị) = [2, 6], left[5] = 4
[-1, 0, -1, 2, -1, 4]


Ghi nhớ: **điều kiện pop quyết định "ai bị loại"**. Ở Cách 4, ta giữ nguyên khung code này nhưng thay điều kiện `nums[j] >= x` bằng "`nums[j]` chứa mọi bit của `x`".

Một cú pháp nhỏ sẽ dùng khi duyệt **ngược** từ phải sang trái: `range(start, stop, step)` với `step = -1`.

```python
range(n - 1, -1, -1)   # n-1, n-2, ..., 0   (stop = -1 không được lấy)
```

In [11]:
n = 6
print(list(range(n - 1, -1, -1)))

[5, 4, 3, 2, 1, 0]


## 3.7 Kỹ thuật đóng góp (contribution technique)

Thay vì hỏi "mảng con này có tốt không?" cho **từng** mảng con (2·10^10 câu hỏi), ta:

1. **Gán mỗi mảng con tốt cho đúng một phần tử đại diện** bên trong nó.
2. Với mỗi phần tử `i`, đếm "nó đại diện cho bao nhiêu mảng con".
3. Cộng tất cả lại.

Nếu đầu trái `l` được chọn tự do trong khoảng `(L, i]` (có `i - L` lựa chọn) và đầu phải `r` tự do trong `[i, R)` (có `R - i` lựa chọn), **độc lập** với nhau, thì số mảng con là **tích**:

```text
(i - L) * (R - i)
```

Ví dụ đơn giản nhất: trong mảng dài `n`, có bao nhiêu mảng con **chứa** chỉ số `i`? Đầu trái `l ∈ [0, i]` (tức `L = -1`, có `i + 1` lựa chọn), đầu phải `r ∈ [i, n-1]` (tức `R = n`, có `n - i` lựa chọn). Vậy có `(i + 1) * (n - i)` mảng con.

In [12]:
n = 5
for i in range(n):
    by_hand = sum(1 for l in range(n) for r in range(l, n) if l <= i <= r)
    formula = (i - (-1)) * (n - i)
    print(f"i={i}: đếm trực tiếp = {by_hand}, công thức (i+1)*(n-i) = {formula}")

i=0: đếm trực tiếp = 5, công thức (i+1)*(n-i) = 5
i=1: đếm trực tiếp = 8, công thức (i+1)*(n-i) = 8
i=2: đếm trực tiếp = 9, công thức (i+1)*(n-i) = 9
i=3: đếm trực tiếp = 8, công thức (i+1)*(n-i) = 8
i=4: đếm trực tiếp = 5, công thức (i+1)*(n-i) = 5


## 3.8 Phá hòa (tie-breaking) khi có phần tử trùng nhau

Xét `[1, 1, 1]`. Mảng con `[1, 1]` có **hai** vị trí cùng giữ giá trị min. Nếu cả hai vị trí đều "nhận" mảng con này làm của mình, ta sẽ **đếm trùng**.

Quy tắc phá hòa: **chọn vị trí xuất hiện cuối cùng (bên phải nhất) của min làm đại diện**. Khi đó:

- Phía **bên trái** của đại diện **được phép** có phần tử bằng min (chúng nằm bên trái nên không phải "cuối cùng").
- Phía **bên phải** của đại diện **không được** có phần tử bằng min (nếu có, đại diện phải là phần tử đó).

Đây là sự **bất đối xứng** cố ý: "trái cho phép bằng, phải cấm bằng". Cell dưới cho thấy mỗi mảng con có đúng một đại diện.

In [13]:
nums = [1, 1, 1]
n = len(nums)
owner_count = [0] * n
for l in range(n):
    for r in range(l, n):
        sub = nums[l:r + 1]
        m = min(sub)
        owner = max(j for j in range(l, r + 1) if nums[j] == m)   # vị trí min bên phải nhất
        owner_count[owner] += 1
        print(f"sub = nums[{l}..{r}] = {sub}, min = {m}, đại diện = chỉ số {owner}")

print("Mỗi chỉ số đại diện cho:", owner_count, "| tổng =", sum(owner_count))

sub = nums[0..0] = [1], min = 1, đại diện = chỉ số 0
sub = nums[0..1] = [1, 1], min = 1, đại diện = chỉ số 1
sub = nums[0..2] = [1, 1, 1], min = 1, đại diện = chỉ số 2
sub = nums[1..1] = [1], min = 1, đại diện = chỉ số 1
sub = nums[1..2] = [1, 1], min = 1, đại diện = chỉ số 2
sub = nums[2..2] = [1], min = 1, đại diện = chỉ số 2
Mỗi chỉ số đại diện cho: [1, 2, 3] | tổng = 6


# 4. Cách 1 — Brute force theo đúng định nghĩa

## Ý tưởng

Làm đúng như đề nói:

1. Duyệt mọi cặp `(l, r)` với `l <= r`.
2. Tính AND và min của `nums[l..r]` **từ đầu**.
3. Nếu bằng nhau thì đếm.

Cách này **hiển nhiên đúng** vì nó chính là định nghĩa. Ta sẽ dùng nó làm "trọng tài" để kiểm tra mọi cách nhanh hơn ở phía sau.

In [14]:
def count_consistent_bruteforce(nums: List[int]) -> int:
    n = len(nums)
    count = 0
    for l in range(n):
        for r in range(l, n):
            and_value = nums[l]
            min_value = nums[l]
            for k in range(l + 1, r + 1):      # tính lại từ đầu cho MỖI mảng con
                and_value &= nums[k]
                min_value = min(min_value, nums[k])
            if and_value == min_value:
                count += 1
    return count


for nums, expected in examples:
    result = count_consistent_bruteforce(nums)
    print(f"{nums} -> {result} (mong đợi {expected})")
    assert result == expected

print("[2, 1, 0] ->", count_consistent_bruteforce([2, 1, 0]))   # mục 2.3: phải ra 5

[5, 3, 1, 2] -> 6 (mong đợi 6)
[1, 1, 1] -> 6 (mong đợi 6)
[0] -> 1 (mong đợi 1)
[2, 1, 0] -> 5


## Cách 1 tốn bao nhiêu công?

Với mỗi mảng con dài `len`, vòng `k` chạm `len` phần tử. Tổng số lần chạm = tổng độ dài mọi mảng con:

```text
n·(n+1)·(n+2)/6  ≈  n³/6
```

Với `n = 2·10^5` con số này cỡ `10^15` — máy tính cần nhiều năm. Cách 1 chỉ dùng được cho `n` rất nhỏ (để kiểm thử).

In [15]:
def brute_work(n):
    # số phần tử được "chạm" = tổng độ dài của mọi mảng con
    return sum(r - l + 1 for l in range(n) for r in range(l, n))


for n in [4, 10, 100]:
    print(f"n={n}: đếm thật = {brute_work(n)}, công thức n(n+1)(n+2)/6 = {n * (n + 1) * (n + 2) // 6}")

n = 2 * 10**5
print("n = 2*10^5 ->", n * (n + 1) * (n + 2) // 6, "lần chạm phần tử")

n=4: đếm thật = 20, công thức n(n+1)(n+2)/6 = 20
n=10: đếm thật = 220, công thức n(n+1)(n+2)/6 = 220
n=100: đếm thật = 171700, công thức n(n+1)(n+2)/6 = 171700
n = 2*10^5 -> 1333353333400000 lần chạm phần tử


# 5. Cách 2 — Cố định đầu trái, mở rộng đầu phải

## Vấn đề của Cách 1

Mảng con `nums[l..r]` và `nums[l..r+1]` chỉ khác nhau **một** phần tử, vậy mà Cách 1 tính lại AND và min từ đầu.

## Ý tưởng

Cố định `l`, cho `r` chạy từ `l` sang phải và **cập nhật dần**:

```python
and_value = and_value & nums[r]        # AND mới = AND cũ & phần tử mới
min_value = min(min_value, nums[r])    # min mới = min(min cũ, phần tử mới)
```

Mỗi bước chỉ tốn `O(1)`.

## Cảnh báo: KHÔNG được dừng sớm

Có thể bạn nghĩ: "gặp mảng con không consistent thì `break` cho nhanh". Sai! Ví dụ `[2, 1, 0]`: `[2, 1]` không consistent nhưng `[2, 1, 0]` lại consistent (số 0 kéo min xuống bằng AND). Mục "Lỗi thường gặp" sẽ chạy thử phiên bản sai này.

In [16]:
def count_consistent_extend(nums: List[int]) -> int:
    n = len(nums)
    count = 0
    for l in range(n):
        and_value = nums[l]
        min_value = nums[l]
        for r in range(l, n):
            and_value &= nums[r]                  # nums[l] & nums[l] = nums[l], vẫn đúng khi r = l
            min_value = min(min_value, nums[r])
            if and_value == min_value:
                count += 1
    return count


for nums, expected in examples:
    result = count_consistent_extend(nums)
    print(f"{nums} -> {result} (mong đợi {expected})")
    assert result == expected

assert count_consistent_extend([2, 1, 0]) == 5
print("[2, 1, 0] ->", count_consistent_extend([2, 1, 0]))

[5, 3, 1, 2] -> 6 (mong đợi 6)
[1, 1, 1] -> 6 (mong đợi 6)
[0] -> 1 (mong đợi 1)
[2, 1, 0] -> 5


## Độ phức tạp của Cách 2

Số bước = số mảng con = `n·(n+1)/2`. Với `n = 2·10^5` là khoảng `2·10^10` bước: vẫn quá chậm.

Cell dưới đo thời gian thật với `n = 1500` rồi nhân tỉ lệ `(2·10^5 / 1500)²` để ước lượng. Con số in ra phụ thuộc máy, nhưng luôn là **nhiều phút**, trong khi LeetCode chỉ cho vài giây.

In [17]:
import random
import time

random.seed(0)
big = [random.randint(0, 10**6) for _ in range(1500)]
t0 = time.perf_counter()
count_consistent_extend(big)
elapsed = time.perf_counter() - t0
scale = (2 * 10**5 / 1500) ** 2
print(f"n = 1500: {elapsed:.3f} giây")
print(f"Ước lượng cho n = 2*10^5: khoảng {elapsed * scale / 60:.0f} phút")

n = 1500: 0.055 giây
Ước lượng cho n = 2*10^5: khoảng 16 phút


# 6. Bước đột phá: hiểu bản chất của "AND == min"

Muốn nhanh hơn `O(n²)`, ta phải hiểu **khi nào** AND bằng min, thay vì thử từng mảng con.

## Đột phá 1: AND luôn `<=` min

Lý do từng bước:

1. AND chỉ giữ những bit mà **mọi** phần tử cùng có, nên AND là **submask của mọi phần tử** (mỗi phần tử `v` đều chứa mọi bit của AND).
2. Theo (F1) ở mục 3.4: `v ⊇ AND` kéo theo `v >= AND`, với **mọi** phần tử `v`.
3. Đặc biệt với phần tử nhỏ nhất: `min >= AND`.

Vậy "AND == min" là trường hợp **AND đạt giá trị lớn nhất có thể**. Ta kiểm chứng trên dữ liệu ngẫu nhiên:

In [18]:
random.seed(42)
checked = 0
violations = 0
for _ in range(200):
    arr = [random.randint(0, 63) for _ in range(random.randint(1, 8))]
    for l in range(len(arr)):
        for r in range(l, len(arr)):
            sub = arr[l:r + 1]
            a = and_of(sub)
            checked += 1
            if not (a <= min(sub) and all(contains(v, a) for v in sub)):
                violations += 1

print("Số mảng con đã kiểm tra:", checked)
print("Số lần vi phạm 'AND <= min' hoặc 'AND là submask của mọi phần tử':", violations)

Số mảng con đã kiểm tra: 3044
Số lần vi phạm 'AND <= min' hoặc 'AND là submask của mọi phần tử': 0


## Đột phá 2: ba điều kiện tương đương

Xét một mảng con `S` có min là `m` và AND là `a`. Ba câu sau **tương đương** (đúng cùng lúc, sai cùng lúc):

- **(A)** `a == m` — tức `S` consistent.
- **(B)** **Mọi phần tử** của `S` đều chứa mọi bit của `m`.
- **(C)** Giá trị `a` **xuất hiện** như một phần tử của `S`.

Chứng minh bằng lời:

- **(A) ⇒ (B)**: `a` là submask của mọi phần tử (Đột phá 1), mà `a == m`, nên `m` là submask của mọi phần tử.
- **(B) ⇒ (A)**: mọi phần tử đều có đủ bit của `m`, nên phép AND giữ nguyên các bit đó: `a ⊇ m`, suy ra `a >= m`. Kết hợp `a <= m` (Đột phá 1) được `a == m`.
- **(A) ⇒ (C)**: min luôn là một phần tử của `S`, mà `a == m`.
- **(C) ⇒ (A)**: `a = S[p]` với một vị trí `p` nào đó, nên `a >= m` (mọi phần tử `>= m`). Kết hợp `a <= m` được `a == m`.

Hai hệ quả sẽ thành hai lời giải nhanh:

- **(B)** → **Cách 4**: "mọi phần tử chứa bit của min" — một điều kiện kiểm tra **từng phần tử riêng lẻ**.
- **(C)** → **Cách 3**: "giá trị AND có mặt trong mảng con" — chỉ cần biết vị trí xuất hiện của một giá trị.

In [19]:
random.seed(7)
total = 0
agree = 0
for _ in range(300):
    arr = [random.randint(0, 15) for _ in range(random.randint(1, 7))]
    for l in range(len(arr)):
        for r in range(l, len(arr)):
            sub = arr[l:r + 1]
            a, m = and_of(sub), min(sub)
            A = (a == m)
            B = all(contains(v, m) for v in sub)
            C_ = a in sub
            total += 1
            agree += (A == B == C_)

print(f"(A), (B), (C) cùng đúng hoặc cùng sai ở {agree}/{total} mảng con")

(A), (B), (C) cùng đúng hoặc cùng sai ở 3359/3359 mảng con


## Đột phá 3: cố định đầu phải, AND chỉ có rất ít giá trị khác nhau

Cố định `r`, cho đầu trái `l` lùi dần từ `r` về `0`:

```text
AND(l..r) = nums[l] & AND(l+1..r)
```

Theo (P2), mỗi lần lùi `l`, AND **giữ nguyên hoặc mất ít nhất một bit**. Bắt đầu từ `nums[r]` có không quá 19 bit `1` (số `<= 10^6` có tối đa 19 bit `1`), nên dãy AND chỉ đổi giá trị **không quá 19 lần** → có **không quá 20 giá trị khác nhau**.

Hệ quả: với mỗi `r`, các đầu trái `l` chia thành **không quá 20 nhóm liên tiếp**, trong mỗi nhóm AND như nhau. Đây là mẹo nổi tiếng cho các bài AND/OR/GCD trên mảng con (thường gọi vui là "log trick").

In [20]:
nums = [12, 7, 13, 15, 14, 15]
r = len(nums) - 1
acc = nums[r]
for l in range(r, -1, -1):
    acc &= nums[l]
    print(f"l={l}: AND(nums[{l}..{r}]) = {acc:2d} = {format(acc, '04b')}")

l=5: AND(nums[5..5]) = 15 = 1111
l=4: AND(nums[4..5]) = 14 = 1110
l=3: AND(nums[3..5]) = 14 = 1110
l=2: AND(nums[2..5]) = 12 = 1100
l=1: AND(nums[1..5]) =  4 = 0100
l=0: AND(nums[0..5]) =  4 = 0100


In [21]:
def max_distinct_and_per_right_end(nums):
    best = 0
    for r in range(len(nums)):
        values = set()
        acc = nums[r]
        for l in range(r, -1, -1):
            acc &= nums[l]
            values.add(acc)
        best = max(best, len(values))
    return best


random.seed(1)
rand_arr = [random.randint(0, 10**6) for _ in range(300)]
FULL19 = (1 << 19) - 1                      # 524287: 19 bit 1, vẫn <= 10^6
block = [FULL19 ^ (1 << b) for b in range(19)] + [FULL19]
hard_arr = block * 15                       # dựng cố ý để AND mất từng bit một

print("Mảng ngẫu nhiên, nhiều giá trị AND khác nhau nhất:", max_distinct_and_per_right_end(rand_arr))
print("Mảng dựng cố ý (xấu nhất):", max_distinct_and_per_right_end(hard_arr))
print("Số bit 1 nhiều nhất của một số <= 10^6:", max(bin(v).count('1') for v in range(10**6 + 1)))

Mảng ngẫu nhiên, nhiều giá trị AND khác nhau nhất: 7
Mảng dựng cố ý (xấu nhất): 20


Số bit 1 nhiều nhất của một số <= 10^6: 19


# 7. Cách 3 — Nhóm các đầu trái theo giá trị AND

## Ý tưởng

Duyệt `r` từ trái sang phải. Duy trì danh sách `groups`, mỗi phần tử là `[a, lo, hi]` với ý nghĩa:

> Với **mọi** `l` trong `[lo, hi]`: `AND(nums[l..r]) = a`.

Theo Đột phá 3, `groups` có không quá 20 nhóm.

Theo Đột phá 2 **(C)**: mảng con `nums[l..r]` (có AND = `a`) consistent **khi và chỉ khi** giá trị `a` xuất hiện trong `nums[l..r]`. Gọi `p = last[a]` là vị trí xuất hiện **cuối cùng** của `a` tính tới `r` (`-1` nếu chưa có). Khi đó:

```text
a xuất hiện trong nums[l..r]   <=>   p >= l
```

Vậy trong nhóm `[a, lo, hi]`, các `l` tốt là `l ∈ [lo, min(hi, p)]`:

| Trường hợp | Số `l` tốt trong nhóm |
|---|---|
| `p < lo` (`a` không có trong đoạn nào của nhóm) | `0` |
| `lo <= p < hi` (chỉ những `l <= p` mới chứa `a`) | `p - lo + 1` |
| `p >= hi` (mọi `l` của nhóm đều chứa `a`) | `hi - lo + 1` |

Gộp lại một công thức: `min(hi, p) - lo + 1` nếu `p >= lo`, ngược lại `0`.

## Cập nhật `groups` khi đi từ `r - 1` sang `r`

Gọi `x = nums[r]`.

1. Mỗi nhóm cũ `[a, lo, hi]` thành `[a & x, lo, hi]`, vì `AND(l..r) = AND(l..r-1) & x`.
2. Hai nhóm **kề nhau** có cùng giá trị mới thì **gộp** lại (nối khoảng `l`).
3. Thêm nhóm mới `[x, r, r]` cho `l = r` (gộp vào nhóm cuối nếu nhóm cuối cũng có giá trị `x`).

Các nhóm luôn xếp theo `lo` tăng dần. Vì AND của đoạn ngắn hơn luôn chứa bit của AND đoạn dài hơn, các giá trị bằng nhau luôn **nằm kề nhau**, nên chỉ cần so với nhóm cuối của danh sách mới.

Ví dụ với `nums = [1, 3, 5, 1, 2]`, đang ở `r = 1` có `groups = [[1, 0, 0], [3, 1, 1]]`. Sang `r = 2` với `x = 5`:

```text
[1, 0, 0] -> [1 & 5, 0, 0] = [1, 0, 0]
[3, 1, 1] -> [3 & 5, 1, 1] = [1, 1, 1]   -> cùng giá trị 1 với nhóm trước -> gộp thành [1, 0, 1]
thêm [5, 2, 2]
=> groups = [[1, 0, 1], [5, 2, 2]]
```

## Mô phỏng Cách 3 trên `nums = [1, 3, 5, 1, 2]` (đáp án 10)

| r | x | groups `[a, lo, hi]` sau cập nhật | `last` của từng `a` | Số `l` tốt từng nhóm | Cộng thêm | Tổng |
|---:|---:|---|---|---|---:|---:|
| 0 | 1 | `[1,0,0]` | `last[1]=0` | a=1: `min(0,0)-0+1 = 1` | 1 | 1 |
| 1 | 3 | `[1,0,0]`, `[3,1,1]` | `last[1]=0`, `last[3]=1` | 1; 1 | 2 | 3 |
| 2 | 5 | `[1,0,1]`, `[5,2,2]` | `last[1]=0`, `last[5]=2` | a=1: `min(1,0)-0+1 = 1` (chỉ `l=0`); a=5: 1 | 2 | 5 |
| 3 | 1 | `[1,0,3]` | `last[1]=3` | a=1: `min(3,3)-0+1 = 4` | 4 | 9 |
| 4 | 2 | `[0,0,3]`, `[2,4,4]` | `last[0]=-1`, `last[2]=4` | a=0: `-1 < 0` → 0; a=2: 1 | 1 | 10 |

Dòng `r = 2` thể hiện trường hợp "một phần": `l = 1` cho mảng con `[3, 5]` có AND = 1 nhưng **không chứa** số 1 (min là 3) nên không consistent; chỉ `l = 0` (mảng `[1, 3, 5]`) là tốt.

Dòng `r = 4` thể hiện trường hợp "không có gì": AND = 0 nhưng số 0 không xuất hiện.

Hàm `trace_and_groups` in đúng các trạng thái này:

In [22]:
def trace_and_groups(nums: List[int]) -> int:
    groups = []                    # mỗi phần tử: [a, lo, hi]
    last = {}
    total = 0
    for r, x in enumerate(nums):
        last[x] = r
        new_groups = []
        for a, lo, hi in groups:
            a &= x
            if new_groups and new_groups[-1][0] == a:
                new_groups[-1][2] = hi             # gộp với nhóm kề trước
            else:
                new_groups.append([a, lo, hi])
        if new_groups and new_groups[-1][0] == x:
            new_groups[-1][2] = r
        else:
            new_groups.append([x, r, r])
        groups = new_groups

        parts = []
        added = 0
        for a, lo, hi in groups:
            p = last.get(a, -1)
            good = min(hi, p) - lo + 1 if p >= lo else 0
            added += good
            parts.append(f"[a={a}, l∈[{lo},{hi}], last={p}] -> {good}")
        total += added
        print(f"r={r}, x={x}: " + "; ".join(parts) + f" | +{added} => tổng {total}")
    return total


trace_and_groups([1, 3, 5, 1, 2])

r=0, x=1: [a=1, l∈[0,0], last=0] -> 1 | +1 => tổng 1
r=1, x=3: [a=1, l∈[0,0], last=0] -> 1; [a=3, l∈[1,1], last=1] -> 1 | +2 => tổng 3
r=2, x=5: [a=1, l∈[0,1], last=0] -> 1; [a=5, l∈[2,2], last=2] -> 1 | +2 => tổng 5
r=3, x=1: [a=1, l∈[0,3], last=3] -> 4 | +4 => tổng 9
r=4, x=2: [a=0, l∈[0,3], last=-1] -> 0; [a=2, l∈[4,4], last=4] -> 1 | +1 => tổng 10


10

## Code gọn của Cách 3

Bỏ phần `print`, ta có hàm chính. Lưu ý `last[x] = r` phải cập nhật **trước** khi đếm, vì mảng con `nums[l..r]` có chứa chính `nums[r]`.

In [23]:
def count_consistent_and_groups(nums: List[int]) -> int:
    groups = []                    # [a, lo, hi]: với l trong [lo, hi], AND(nums[l..r]) = a
    last = {}                      # last[v] = vị trí xuất hiện cuối cùng của v (tính tới r)
    total = 0
    for r, x in enumerate(nums):
        last[x] = r
        new_groups = []
        for a, lo, hi in groups:
            a &= x
            if new_groups and new_groups[-1][0] == a:
                new_groups[-1][2] = hi
            else:
                new_groups.append([a, lo, hi])
        if new_groups and new_groups[-1][0] == x:
            new_groups[-1][2] = r
        else:
            new_groups.append([x, r, r])
        groups = new_groups

        for a, lo, hi in groups:
            p = last.get(a, -1)
            if p >= lo:
                total += min(hi, p) - lo + 1
    return total


for nums, expected in examples:
    result = count_consistent_and_groups(nums)
    print(f"{nums} -> {result} (mong đợi {expected})")
    assert result == expected

assert count_consistent_and_groups([2, 1, 0]) == 5
assert count_consistent_and_groups([1, 3, 5, 1, 2]) == 10
print("Các test của Cách 3 PASS")

[5, 3, 1, 2] -> 6 (mong đợi 6)
[1, 1, 1] -> 6 (mong đợi 6)
[0] -> 1 (mong đợi 1)
Các test của Cách 3 PASS


## Độ phức tạp của Cách 3

Gọi `B` là số nhóm tối đa (`B <= 20` theo Đột phá 3).

- **Thời gian**: mỗi `r` xử lý `<= B` nhóm → `O(n · B)`, khoảng `4·10^6` thao tác với `n = 2·10^5`. Chạy được trên LeetCode bằng Python.
- **Bộ nhớ**: `O(B)` cho `groups` + `O(D)` cho `last`, với `D` là số giá trị khác nhau (`D <= n`).

Cell dưới đo thời gian với `n = 2·10^5` (kết quả phụ thuộc máy, thường dưới một giây):

In [24]:
random.seed(2026)
big = [random.randint(0, 10**6) for _ in range(2 * 10**5)]
t0 = time.perf_counter()
ans_groups = count_consistent_and_groups(big)
print(f"Cách 3, n = 2*10^5: đáp án = {ans_groups}, thời gian = {time.perf_counter() - t0:.3f} giây")

Cách 3, n = 2*10^5: đáp án = 4995799872, thời gian = 0.225 giây


# 8. Cách 4 — Đóng góp của từng phần tử + monotonic stack (`O(n)`)

Cách 3 đã đủ nhanh, nhưng còn hệ số `B ≈ 20`. Cách 4 dùng Đột phá 2 **(B)** để đạt `O(n)` gọn gàng.

## 8.1 Gán mỗi mảng con consistent cho một đại diện

Mỗi mảng con consistent có một min `m`. Chọn **đại diện** là vị trí **bên phải nhất** của `m` trong mảng con (quy tắc phá hòa ở mục 3.8).

Cố định một chỉ số `i`, đặt `x = nums[i]`. Mảng con `nums[l..r]` (với `l <= i <= r`) **consistent và nhận `i` làm đại diện** khi và chỉ khi:

1. **Mọi** phần tử trong `nums[l..r]` đều chứa mọi bit của `x` (Đột phá 2 **(B)**), và
2. **Không** phần tử nào trong `nums[i+1..r]` bằng `x` (để `i` là vị trí min bên phải nhất).

Chiều ngược lại cũng đúng: nếu 1 và 2 thỏa thì mọi phần tử `>= x` (F1) nên min = `x`; AND chứa mọi bit của `x` mà lại `<= x` nên AND = `x` → consistent, và `i` là vị trí min cuối cùng.

## 8.2 Tách thành hai phía độc lập

- **Phía trái**: mọi `nums[j]` với `l <= j < i` phải chứa `x`. Gọi `L` = chỉ số `j < i` **gần nhất** mà `nums[j]` **không** chứa `x` (không có thì `L = -1`). Khi đó `l ∈ (L, i]` → có `i - L` lựa chọn.
- **Phía phải**: mọi `nums[j]` với `i < j <= r` phải chứa `x` **và khác `x`**. Gọi `R` = chỉ số `j > i` gần nhất mà `nums[j]` không chứa `x` **hoặc** `nums[j] == x` (không có thì `R = n`). Khi đó `r ∈ [i, R)` → có `R - i` lựa chọn.

Hai phía độc lập nên:

```text
đóng góp của i = (i - L) * (R - i)
đáp án         = tổng đóng góp của mọi i
```

Để ý sự bất đối xứng: bên trái **cho phép** gặp lại `x` (vì `x ⊇ x`), bên phải thì `x` là **vật cản**.

## 8.3 Bước 4a — Tìm `L`, `R` bằng cách "đi bộ" (chậm, để kiểm chứng công thức)

Trước khi tối ưu, ta viết bản dễ hiểu: từ `i` đi lùi sang trái cho tới khi gặp vật cản, rồi đi tới sang phải. Mục đích: **kiểm chứng công thức đóng góp** trên ví dụ 1.

In [25]:
def count_consistent_expand(nums: List[int], verbose: bool = False) -> int:
    n = len(nums)
    total = 0
    for i, x in enumerate(nums):
        L = i - 1
        while L >= 0 and (nums[L] & x) == x:                    # nums[L] chứa x -> đi tiếp
            L -= 1
        R = i + 1
        while R < n and nums[R] != x and (nums[R] & x) == x:    # chứa x VÀ khác x -> đi tiếp
            R += 1
        contribution = (i - L) * (R - i)
        total += contribution
        if verbose:
            print(f"i={i}, x={x}: L={L}, R={R}, đóng góp = ({i}-({L})) * ({R}-{i}) = {contribution}")
    return total


print("Tổng =", count_consistent_expand([5, 3, 1, 2], verbose=True))

i=0, x=5: L=-1, R=1, đóng góp = (0-(-1)) * (1-0) = 1
i=1, x=3: L=0, R=2, đóng góp = (1-(0)) * (2-1) = 1
i=2, x=1: L=-1, R=3, đóng góp = (2-(-1)) * (3-2) = 3
i=3, x=2: L=2, R=4, đóng góp = (3-(2)) * (4-3) = 1
Tổng = 6


Đọc kết quả: `i = 2` (giá trị 1) có `L = -1`, `R = 3`, đóng góp `3 * 1 = 3`. Đó chính là ba mảng con `[1]`, `[3, 1]`, `[5, 3, 1]` — đúng ba mảng con consistent có min bằng 1 trong bảng mục 2.1. Ba chỉ số còn lại mỗi chỉ số đóng góp 1 (mảng con một phần tử). Tổng 6.

### Vì sao bản "đi bộ" vẫn chậm?

Với mảng toàn số 0: mọi phần tử đều chứa 0, nên `L` của mỗi `i` phải lùi tới tận `-1` → tổng `n(n-1)/2` bước, tức `O(n²)`.

In [26]:
def expand_steps(nums):
    n = len(nums)
    steps = 0
    for i, x in enumerate(nums):
        L = i - 1
        while L >= 0 and (nums[L] & x) == x:
            L -= 1
            steps += 1
        R = i + 1
        while R < n and nums[R] != x and (nums[R] & x) == x:
            R += 1
            steps += 1
    return steps


for n in [1000, 2000]:
    print(f"n={n}, toàn số 0: số bước đi bộ = {expand_steps([0] * n)}  (n(n-1)/2 = {n * (n - 1) // 2})")

n=1000, toàn số 0: số bước đi bộ = 499500  (n(n-1)/2 = 499500)
n=2000, toàn số 0: số bước đi bộ = 1999000  (n(n-1)/2 = 1999000)


## 8.4 Bước 4b — Tìm `L` bằng stack (duyệt trái → phải)

So với bài "phần tử nhỏ hơn gần nhất bên trái" (mục 3.6), ta chỉ đổi **điều kiện pop**:

| Bài kinh điển | Bài này |
|---|---|
| pop khi `nums[top] >= x` | pop khi `nums[top]` **chứa mọi bit** của `x`: `(nums[top] & x) == x` |

Quy trình cho từng `i` (với `x = nums[i]`):

1. Trong khi đỉnh stack chứa `x`: pop.
2. Nếu stack còn phần tử: `left[i] = stack[-1]`; nếu rỗng: `left[i] = -1`.
3. Push `i`.

**Vì sao vứt bỏ chỉ số bị pop là an toàn?** Giả sử `j` bị pop bởi `k` (`j < k`, `nums[j] ⊇ nums[k]`). Xét một `i` bất kỳ phía sau `k`:

- Nếu `nums[k]` chứa `x = nums[i]` thì `nums[j]` cũng chứa `x` (**bắc cầu** F2) → `j` không phải vật cản của `i`.
- Nếu `nums[k]` không chứa `x` thì `k` là vật cản và **gần `i` hơn** `j` → `j` không bao giờ là vật cản **gần nhất**.

Hai trường hợp đều cho thấy `j` không bao giờ là đáp án cho ai phía sau. Vì vậy vật cản gần nhất `L` của `i` luôn còn nằm trong stack; mọi chỉ số nằm trên nó đều thuộc `(L, i)` nên chứa `x` và bị pop hết; vòng `while` dừng **đúng tại `L`**.

## 8.5 Bước 4c — Tìm `R` bằng stack (duyệt phải → trái)

Đối xứng, nhưng nhớ quy tắc phá hòa: bên phải, giá trị **bằng `x` là vật cản**. Điều kiện pop:

```python
while stack and nums[stack[-1]] != x and (nums[stack[-1]] & x) == x:
    stack.pop()
```

tức là chỉ pop khi đỉnh **chứa `x` và khác `x`** (supermask "thật sự"). Quan hệ này cũng bắc cầu (F3), nên lập luận "vứt bỏ an toàn" y hệt như trên.

Ở lượt phải → trái, ngay khi có `R` của `i`, ta cộng luôn đóng góp `(i - left[i]) * (R - i)` — không cần lưu mảng `right`.

In [27]:
def count_consistent_stack(nums: List[int]) -> int:
    n = len(nums)

    # Lượt 1: trái -> phải, left[i] = vật cản gần nhất bên trái (không chứa x)
    left = [-1] * n
    stack = []
    for i, x in enumerate(nums):
        while stack and (nums[stack[-1]] & x) == x:
            stack.pop()
        if stack:
            left[i] = stack[-1]
        stack.append(i)

    # Lượt 2: phải -> trái, right = vật cản gần nhất bên phải (không chứa x HOẶC bằng x)
    stack = []
    total = 0
    for i in range(n - 1, -1, -1):
        x = nums[i]
        while stack and nums[stack[-1]] != x and (nums[stack[-1]] & x) == x:
            stack.pop()
        right = stack[-1] if stack else n
        stack.append(i)
        total += (i - left[i]) * (right - i)
    return total


for nums, expected in examples:
    result = count_consistent_stack(nums)
    print(f"{nums} -> {result} (mong đợi {expected})")
    assert result == expected

assert count_consistent_stack([2, 1, 0]) == 5
assert count_consistent_stack([5, 7, 2, 3, 2, 6]) == 17
print("Các test của Cách 4 PASS")

[5, 3, 1, 2] -> 6 (mong đợi 6)
[1, 1, 1] -> 6 (mong đợi 6)
[0] -> 1 (mong đợi 1)
Các test của Cách 4 PASS


## 8.6 Mô phỏng Cách 4 trên `nums = [5, 7, 2, 3, 2, 6]` (đáp án 17)

Nhị phân: `5 = 101`, `7 = 111`, `2 = 010`, `3 = 011`, `6 = 110`. Stack ghi dạng `chỉ_số:giá_trị`, từ đáy lên đỉnh.

### Lượt 1 (trái → phải): tìm `left[i]`

| i | x | Stack trước | Bị pop (vì chứa x) | `left[i]` | Stack sau |
|---:|---|---|---|---:|---|
| 0 | 5 (101) | `[]` | — | -1 | `[0:5]` |
| 1 | 7 (111) | `[0:5]` | — (`5 & 7 = 5 ≠ 7`) | 0 | `[0:5, 1:7]` |
| 2 | 2 (010) | `[0:5, 1:7]` | `1:7` (`7 & 2 = 2`) | 0 (`5 & 2 = 0 ≠ 2`) | `[0:5, 2:2]` |
| 3 | 3 (011) | `[0:5, 2:2]` | — (`2 & 3 = 2 ≠ 3`) | 2 | `[0:5, 2:2, 3:3]` |
| 4 | 2 (010) | `[0:5, 2:2, 3:3]` | `3:3`, `2:2` (bằng nhau vẫn pop) | 0 | `[0:5, 4:2]` |
| 5 | 6 (110) | `[0:5, 4:2]` | — (`2 & 6 = 2 ≠ 6`) | 4 | `[0:5, 4:2, 5:6]` |

Điểm đáng chú ý ở `i = 2`: `5 > 2` nhưng `5` **không chứa** bit của `2`, nên `5` là vật cản. Một monotonic stack theo số học thông thường sẽ bỏ qua `5` và cho kết quả sai.

### Lượt 2 (phải → trái): tìm `right`, cộng đóng góp

| i | x | Stack trước | Bị pop (chứa x và ≠ x) | `right` | Đóng góp `(i-left)*(right-i)` | Tổng |
|---:|---|---|---|---:|---|---:|
| 5 | 6 | `[]` | — | 6 | `(5-4)*(6-5) = 1` | 1 |
| 4 | 2 | `[5:6]` | `5:6` (`6 & 2 = 2`) | 6 | `(4-0)*(6-4) = 8` | 9 |
| 3 | 3 | `[4:2]` | — | 4 | `(3-2)*(4-3) = 1` | 10 |
| 2 | 2 | `[4:2, 3:3]` | `3:3`; dừng ở `4:2` vì **bằng** x | 4 | `(2-0)*(4-2) = 4` | 14 |
| 1 | 7 | `[4:2, 2:2]` | — | 2 | `(1-0)*(2-1) = 1` | 15 |
| 0 | 5 | `[4:2, 2:2, 1:7]` | `1:7` (`7 & 5 = 5`) | 2 | `(0+1)*(2-0) = 2` | 17 |

Dòng `i = 2` thể hiện phá hòa: số `2` ở chỉ số 4 chặn phía phải của số `2` ở chỉ số 2. Nhờ vậy mảng con như `[2, 3, 2]` chỉ được đếm cho chỉ số 4 (đóng góp 8), không bị đếm hai lần.

In [28]:
def trace_stack(nums: List[int]) -> int:
    n = len(nums)
    width = max(1, max(nums).bit_length())
    fmt = lambda st: "[" + ", ".join(f"{j}:{nums[j]}" for j in st) + "]"

    print("--- Lượt 1: trái -> phải (tìm left) ---")
    left = [-1] * n
    stack = []
    for i, x in enumerate(nums):
        before = fmt(stack)
        popped = []
        while stack and (nums[stack[-1]] & x) == x:
            popped.append(stack.pop())
        if stack:
            left[i] = stack[-1]
        stack.append(i)
        print(f"i={i} x={x} ({format(x, f'0{width}b')}) | trước {before} | pop {fmt(popped)} "
              f"| left[{i}]={left[i]} | sau {fmt(stack)}")

    print("--- Lượt 2: phải -> trái (tìm right, cộng đóng góp) ---")
    stack = []
    total = 0
    for i in range(n - 1, -1, -1):
        x = nums[i]
        before = fmt(stack)
        popped = []
        while stack and nums[stack[-1]] != x and (nums[stack[-1]] & x) == x:
            popped.append(stack.pop())
        right = stack[-1] if stack else n
        stack.append(i)
        contribution = (i - left[i]) * (right - i)
        total += contribution
        print(f"i={i} x={x} | trước {before} | pop {fmt(popped)} | right={right} "
              f"| đóng góp ({i}-({left[i]}))*({right}-{i})={contribution} | tổng={total}")
    return total


print("Đáp án:", trace_stack([5, 7, 2, 3, 2, 6]))

--- Lượt 1: trái -> phải (tìm left) ---
i=0 x=5 (101) | trước [] | pop [] | left[0]=-1 | sau [0:5]
i=1 x=7 (111) | trước [0:5] | pop [] | left[1]=0 | sau [0:5, 1:7]
i=2 x=2 (010) | trước [0:5, 1:7] | pop [1:7] | left[2]=0 | sau [0:5, 2:2]
i=3 x=3 (011) | trước [0:5, 2:2] | pop [] | left[3]=2 | sau [0:5, 2:2, 3:3]
i=4 x=2 (010) | trước [0:5, 2:2, 3:3] | pop [3:3, 2:2] | left[4]=0 | sau [0:5, 4:2]
i=5 x=6 (110) | trước [0:5, 4:2] | pop [] | left[5]=4 | sau [0:5, 4:2, 5:6]
--- Lượt 2: phải -> trái (tìm right, cộng đóng góp) ---
i=5 x=6 | trước [] | pop [] | right=6 | đóng góp (5-(4))*(6-5)=1 | tổng=1
i=4 x=2 | trước [5:6] | pop [5:6] | right=6 | đóng góp (4-(0))*(6-4)=8 | tổng=9
i=3 x=3 | trước [4:2] | pop [] | right=4 | đóng góp (3-(2))*(4-3)=1 | tổng=10
i=2 x=2 | trước [4:2, 3:3] | pop [3:3] | right=4 | đóng góp (2-(0))*(4-2)=4 | tổng=14
i=1 x=7 | trước [4:2, 2:2] | pop [] | right=2 | đóng góp (1-(0))*(2-1)=1 | tổng=15
i=0 x=5 | trước [4:2, 2:2, 1:7] | pop [1:7] | right=2 | đóng góp (0-

## 8.7 Độ phức tạp của Cách 4

- Mỗi chỉ số được push **đúng 1 lần** và pop **tối đa 1 lần** trong mỗi lượt → tổng số lần pop `<= 2n`.
- **Thời gian**: `O(n)` (khấu hao).
- **Bộ nhớ**: `O(n)` cho mảng `left` và stack.

So sánh trực tiếp với bản "đi bộ" trên mảng toàn số 0 (`n = 2000`): bản đi bộ tốn `1 999 000` bước, bản stack chỉ tốn `1 999` lần pop.

In [29]:
def stack_pops(nums):
    n = len(nums)
    pops = 0
    stack = []
    for i, x in enumerate(nums):
        while stack and (nums[stack[-1]] & x) == x:
            stack.pop()
            pops += 1
        stack.append(i)
    stack = []
    for i in range(n - 1, -1, -1):
        x = nums[i]
        while stack and nums[stack[-1]] != x and (nums[stack[-1]] & x) == x:
            stack.pop()
            pops += 1
        stack.append(i)
    return pops


zeros = [0] * 2000
print("Đi bộ:", expand_steps(zeros), "bước | Stack:", stack_pops(zeros), "lần pop")

t0 = time.perf_counter()
ans_stack = count_consistent_stack(big)
print(f"Cách 4, n = 2*10^5 ngẫu nhiên: đáp án = {ans_stack}, thời gian = {time.perf_counter() - t0:.3f} giây")
assert ans_stack == ans_groups          # khớp với Cách 3 trên cùng dữ liệu lớn

Đi bộ: 1999000 bước | Stack: 1999 lần pop


Cách 4, n = 2*10^5 ngẫu nhiên: đáp án = 4995799872, thời gian = 0.059 giây


# 9. Phiên bản nộp LeetCode

Chuyển Cách 4 vào đúng class và method mà LeetCode yêu cầu: `countConsistent(self, nums)`. Trên trang đề, chữ ký có thể hiện là `nums: list[int]`; viết `List[int]` (cần `from typing import List`) là tương đương.

Code nộp bài nên gọn: không `print`, không biến thừa.

In [30]:
from typing import List


class Solution:
    def countConsistent(self, nums: List[int]) -> int:
        n = len(nums)
        left = [-1] * n
        stack = []
        for i, x in enumerate(nums):
            while stack and (nums[stack[-1]] & x) == x:
                stack.pop()
            if stack:
                left[i] = stack[-1]
            stack.append(i)

        stack = []
        ans = 0
        for i in range(n - 1, -1, -1):
            x = nums[i]
            while stack and nums[stack[-1]] != x and (nums[stack[-1]] & x) == x:
                stack.pop()
            right = stack[-1] if stack else n
            stack.append(i)
            ans += (i - left[i]) * (right - i)
        return ans

In [31]:
sol = Solution()

tests = [
    ([5, 3, 1, 2], 6),          # ví dụ 1
    ([1, 1, 1], 6),             # ví dụ 2: toàn phần tử trùng
    ([0], 1),                   # ví dụ 3
    ([10**6], 1),               # một phần tử, giá trị lớn nhất
    ([2, 1], 2),                # 2 > 1 nhưng 2 không chứa bit của 1
    ([2, 1, 0], 5),             # số 0 "cứu" mảng con
    ([7, 7, 7, 7], 10),         # toàn bằng nhau: mọi mảng con đều tốt
    ([1, 2, 4, 8], 4),          # không có bit chung: chỉ mảng con 1 phần tử
    ([5, 7, 2, 3, 2, 6], 17),   # ví dụ mô phỏng
    ([1, 3, 5, 1, 2], 10),      # ví dụ mô phỏng Cách 3
]

for nums, expected in tests:
    result = sol.countConsistent(nums)
    print(f"{str(nums):<22} -> {result:>3} | expected = {expected:>3} | {'PASS' if result == expected else 'FAIL'}")
    assert result == expected

n = 2 * 10**5
assert sol.countConsistent([0] * n) == n * (n + 1) // 2      # đáp án vượt 32-bit
assert sol.countConsistent(big) == ans_groups
print("Mảng 2*10^5 số 0 ->", sol.countConsistent([0] * n))
print("Tất cả test PASS!")

[5, 3, 1, 2]           ->   6 | expected =   6 | PASS
[1, 1, 1]              ->   6 | expected =   6 | PASS
[0]                    ->   1 | expected =   1 | PASS
[1000000]              ->   1 | expected =   1 | PASS
[2, 1]                 ->   2 | expected =   2 | PASS
[2, 1, 0]              ->   5 | expected =   5 | PASS
[7, 7, 7, 7]           ->  10 | expected =  10 | PASS
[1, 2, 4, 8]           ->   4 | expected =   4 | PASS
[5, 7, 2, 3, 2, 6]     ->  17 | expected =  17 | PASS
[1, 3, 5, 1, 2]        ->  10 | expected =  10 | PASS


Mảng 2*10^5 số 0 -> 20000100000
Tất cả test PASS!


# 10. Đối chiếu tất cả các cách trên dữ liệu ngẫu nhiên

Ví dụ trong đề quá ít để tin tưởng. Ta sinh nhiều mảng nhỏ ngẫu nhiên (seed cố định để chạy lại ra y hệt) và so **mọi** cách với brute force.

Mẹo sinh dữ liệu: chọn giá trị trong các khoảng nhỏ như `0..3`, `0..7` để có **nhiều phần tử trùng** và **nhiều quan hệ chứa bit** — đó là nơi các lỗi phá hòa hay lộ ra. Thêm khoảng `0..10^6` để có số lớn.

In [32]:
random.seed(2024)
approaches = {
    "extend (Cách 2)": count_consistent_extend,
    "and_groups (Cách 3)": count_consistent_and_groups,
    "expand (Cách 4a)": count_consistent_expand,
    "stack (Cách 4)": count_consistent_stack,
    "Solution": Solution().countConsistent,
}

num_cases = 400
for _ in range(num_cases):
    n = random.randint(1, 10)
    hi = random.choice([1, 3, 7, 15, 10**6])
    arr = [random.randint(0, hi) for _ in range(n)]
    expected = count_consistent_bruteforce(arr)
    for name, fn in approaches.items():
        got = fn(arr)
        assert got == expected, (name, arr, got, expected)

print(f"{num_cases} test ngẫu nhiên: mọi cách đều khớp brute force")

400 test ngẫu nhiên: mọi cách đều khớp brute force


# 11. Vì sao đúng? (tóm tắt chứng minh)

## Cách 3

- **Bất biến (invariant)** sau khi xử lý `r`: `groups` liệt kê, theo thứ tự `lo` tăng, mọi giá trị `AND(nums[l..r])` cùng khoảng `l` cho ra giá trị đó. Đúng vì `AND(l..r) = AND(l..r-1) & nums[r]`, và các nhóm cùng giá trị luôn kề nhau.
- Với mỗi `l`, mảng con `nums[l..r]` có AND = `a` consistent ⇔ `a` xuất hiện trong đoạn (Đột phá 2 **(C)**) ⇔ `last[a] >= l` (vì `last[a] <= r` là lần xuất hiện cuối tính tới `r`).
- Mỗi mảng con `[l..r]` được xét đúng một lần: tại bước `r`, trong đúng một nhóm chứa `l`.

## Cách 4

1. **Không đếm trùng**: mỗi mảng con consistent có duy nhất một vị trí min bên phải nhất → chỉ được đếm ở đúng một `i`.
2. **Không bỏ sót, không đếm nhầm**: với `i` cố định, tập mảng con được đếm ở `i` **đúng bằng** tập `{[l..r] : L < l <= i <= r < R}`, và đó chính xác là các mảng con consistent nhận `i` làm đại diện (mục 8.1–8.2).
3. **Stack trả đúng `L` và `R`**: nhờ tính bắc cầu (F2, F3), một chỉ số đã bị pop không bao giờ là vật cản gần nhất của ai phía sau; vật cản gần nhất luôn còn trong stack và mọi chỉ số nằm trên nó đều bị pop (mục 8.4–8.5).

Ghi nhớ tinh thần: **monotonic stack không chỉ dành cho `<` / `<=`**. Nó dùng được với **mọi quan hệ bắc cầu** (chứa bit, chia hết, ...), miễn là điều kiện pop được chọn đúng.

# 12. Độ phức tạp và so sánh các cách

Ký hiệu: `n = len(nums)`; `B` = số nhóm AND tối đa cho một đầu phải (`B <= 20` vì `nums[i] <= 10^6`); `D` = số giá trị khác nhau trong `nums`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|---|
| 1 | Brute force, tính lại từ đầu | `O(n³)` | `O(1)` | Trọng tài để kiểm thử |
| 2 | Cố định `l`, mở rộng `r` | `O(n²)` | `O(1)` | Dễ viết, vẫn quá chậm |
| 3 | Nhóm theo giá trị AND + `last` | `O(n · B)` | `O(B + D)` | Kỹ thuật chung cho AND/OR/GCD |
| 4a | Đóng góp, tìm `L`/`R` bằng đi bộ | `O(n²)` xấu nhất | `O(1)` | Kiểm chứng công thức |
| 4 | Đóng góp + monotonic stack | `O(n)` | `O(n)` | Lời giải nộp bài |

Về tốc độ thực tế trong Python với `n = 2·10^5`: Cách 4 chạy cỡ vài phần trăm giây, Cách 3 chậm hơn vài lần (do hệ số `B`) nhưng vẫn qua được. Cách 1, 2, 4a không qua được ràng buộc.

# 13. Lỗi thường gặp

## Lỗi 1 — Nhầm "lớn hơn hoặc bằng" với "chứa mọi bit"

Viết monotonic stack quen tay với điều kiện số học `nums[top] >= x` (đếm số mảng con nhận `i` làm min bên phải nhất). Nhưng **mọi** mảng con đều có một min bên phải nhất, nên cách này luôn trả về `n·(n+1)/2` — tức đếm **tất cả** mảng con!

Sửa: điều kiện "đi qua được" phải là `(nums[j] & x) == x`.

In [33]:
def wrong_numeric_stack(nums):
    # SAI: dùng so sánh số học thay vì "chứa mọi bit"
    n = len(nums)
    left = [-1] * n
    stack = []
    for i, x in enumerate(nums):
        while stack and nums[stack[-1]] >= x:
            stack.pop()
        if stack:
            left[i] = stack[-1]
        stack.append(i)
    stack = []
    total = 0
    for i in range(n - 1, -1, -1):
        x = nums[i]
        while stack and nums[stack[-1]] > x:
            stack.pop()
        right = stack[-1] if stack else n
        stack.append(i)
        total += (i - left[i]) * (right - i)
    return total


for nums in ([5, 3, 1, 2], [2, 1]):
    print(f"{nums}: sai = {wrong_numeric_stack(nums)}, đúng = {count_consistent_stack(nums)}, "
          f"n(n+1)/2 = {len(nums) * (len(nums) + 1) // 2}")

[5, 3, 1, 2]: sai = 10, đúng = 6, n(n+1)/2 = 10
[2, 1]: sai = 3, đúng = 2, n(n+1)/2 = 3


## Lỗi 2 — Phá hòa sai: đếm trùng hoặc đếm thiếu

- Nếu **cả hai phía** đều cho phép đi qua phần tử bằng `x` → mảng con có nhiều vị trí min bị đếm nhiều lần (**đếm trùng**).
- Nếu **cả hai phía** đều cấm phần tử bằng `x` → mảng con có nhiều vị trí min không được ai nhận (**đếm thiếu**).

Đáng sợ là ví dụ 1 `[5, 3, 1, 2]` không có phần tử trùng nên **cả hai bản sai đều ra đúng 6**. Chỉ ví dụ 2 `[1, 1, 1]` mới bắt được lỗi.

In [34]:
def contribution_with_rules(nums, allow_equal_left, allow_equal_right):
    n = len(nums)
    total = 0
    for i, x in enumerate(nums):
        L = i - 1
        while L >= 0 and (nums[L] & x) == x and (allow_equal_left or nums[L] != x):
            L -= 1
        R = i + 1
        while R < n and (nums[R] & x) == x and (allow_equal_right or nums[R] != x):
            R += 1
        total += (i - L) * (R - i)
    return total


for nums in ([5, 3, 1, 2], [1, 1, 1]):
    print(f"{nums}:")
    print("   trái cho phép bằng, phải cấm bằng (ĐÚNG):", contribution_with_rules(nums, True, False))
    print("   cả hai phía cho phép bằng (đếm trùng)    :", contribution_with_rules(nums, True, True))
    print("   cả hai phía cấm bằng (đếm thiếu)         :", contribution_with_rules(nums, False, False))

[5, 3, 1, 2]:
   trái cho phép bằng, phải cấm bằng (ĐÚNG): 6
   cả hai phía cho phép bằng (đếm trùng)    : 6
   cả hai phía cấm bằng (đếm thiếu)         : 6
[1, 1, 1]:
   trái cho phép bằng, phải cấm bằng (ĐÚNG): 6
   cả hai phía cho phép bằng (đếm trùng)    : 10
   cả hai phía cấm bằng (đếm thiếu)         : 3


## Lỗi 3 — Dừng sớm (`break`) trong Cách 2

Như đã thấy ở mục 2.2, một mảng con xấu vẫn có thể trở nên tốt khi kéo dài (phần tử mới nhỏ hơn có thể kéo min xuống bằng AND). Phiên bản có `break` cho kết quả sai với `[2, 1, 0]`.

In [35]:
def wrong_extend_with_break(nums):
    n = len(nums)
    count = 0
    for l in range(n):
        and_value = nums[l]
        min_value = nums[l]
        for r in range(l, n):
            and_value &= nums[r]
            min_value = min(min_value, nums[r])
            if and_value == min_value:
                count += 1
            else:
                break                      # SAI: bỏ lỡ [2, 1, 0]
    return count


print("[2, 1, 0]: có break =", wrong_extend_with_break([2, 1, 0]), "| đúng =", count_consistent_extend([2, 1, 0]))

[2, 1, 0]: có break = 4 | đúng = 5


## Lỗi 4 — Thứ tự ưu tiên toán tử khi dịch sang ngôn ngữ khác

Trong Python, `a & x == x` được hiểu là `(a & x) == x` (đúng ý). Nhưng trong C++/Java/JavaScript, `==` được tính **trước** `&`, nên cùng dòng đó thành `a & (x == x)` = `a & 1` — một lỗi rất khó thấy. Cell dưới mô phỏng hai cách hiểu bằng Python.

## Lỗi 5 — Tràn số khi dịch sang ngôn ngữ khác

Đáp án có thể tới `20 000 100 000` (mảng `2·10^5` số 0), vượt `int` 32-bit. Trong Java/C++ phải dùng `long` / `long long` cho biến `ans` **và** cho phép nhân `(i - left[i]) * (right - i)`.

## Lỗi 6 — Quên cập nhật `last[x] = r` trước khi đếm (Cách 3)

Nếu cập nhật `last` sau khi đếm, mảng con `[x]` (và mọi mảng con mà `x` chỉ xuất hiện ở `r`) sẽ không được đếm.

In [36]:
a, x = 6, 2
print("Python hiểu  (a & x) == x :", (a & x) == x)
print("C/Java hiểu  a & (x == x) :", a & (x == x), "-> giá trị 0 tức là 'sai' trong C")

print("Đáp án lớn nhất có thể:", 2 * 10**5 * (2 * 10**5 + 1) // 2, "> int32 =", 2**31 - 1)

Python hiểu  (a & x) == x : True
C/Java hiểu  a & (x == x) : 0 -> giá trị 0 tức là 'sai' trong C
Đáp án lớn nhất có thể: 20000100000 > int32 = 2147483647


## Các edge case nên tự test

| Input | Đáp án | Vì sao đáng test |
|---|---:|---|
| `[0]` | 1 | Một phần tử, giá trị 0 |
| `[2, 1]` | 2 | `2 > 1` nhưng `2` không chứa bit của `1` |
| `[2, 1, 0]` | 5 | Số 0 làm mọi mảng con chứa nó consistent; bẫy `break` |
| `[1, 1, 1]` | 6 | Phần tử trùng: bẫy phá hòa |
| `[1, 2, 4, 8]` | 4 | Không có bit chung: chỉ mảng con một phần tử |
| `[0] * 200000` | 20000100000 | Đáp án vượt 32-bit; bản "đi bộ" bị `O(n²)` |

# 14. Dấu hiệu nhận dạng và bài liên quan

## Dấu hiệu 1: "AND / OR / GCD của mảng con" so với "min / max"

- AND `<=` min, OR `>=` max, GCD `<=` min.
- Dấu bằng xảy ra ⇔ phần tử cực trị có **quan hệ đặc biệt với mọi phần tử** (chứa bit / bị chứa bit / chia hết). Hãy tìm quan hệ đó trước tiên.

## Dấu hiệu 2: "AND / OR / GCD của các mảng con kết thúc tại `r`"

- Chỉ có `O(log(max))` giá trị khác nhau → duy trì danh sách nhóm như Cách 3 ("log trick").

## Dấu hiệu 3: "đếm / tính tổng trên mọi mảng con" mà điều kiện xoay quanh min / max

- Nghĩ ngay tới **kỹ thuật đóng góp**: gán mỗi mảng con cho một phần tử đại diện, đếm `(i - L) * (R - i)`.
- Luôn hỏi: "có phần tử trùng không?" → chọn quy tắc phá hòa bất đối xứng.

## Dấu hiệu 4: "phần tử gần nhất bên trái / phải không thỏa quan hệ X"

- Nếu X **bắc cầu** → monotonic stack, điều kiện pop là "đỉnh thỏa X với phần tử mới".

## Bài liên quan nên luyện

| Bài | Kỹ thuật chung với bài này |
|---|---|
| 907. Sum of Subarray Minimums | Đóng góp + monotonic stack + phá hòa |
| 2104. Sum of Subarray Ranges | Đóng góp cho cả min và max |
| 84. Largest Rectangle in Histogram | Phần tử nhỏ hơn gần nhất hai phía |
| 496 / 503. Next Greater Element I / II | Monotonic stack cơ bản |
| 898. Bitwise ORs of Subarrays | Log trick với OR |
| 1521. Find a Value of a Mysterious Function Closest to Target | Log trick với AND |
| 3209. Number of Subarrays With AND Value of K | Log trick với AND, đếm mảng con |
| 2447. Number of Subarrays With GCD Equal to K | Ý tưởng tương tự với GCD |

Cell dưới áp dụng **đúng khung code** của Cách 4 cho bài 907 (tổng min của mọi mảng con, bản không lấy modulo): chỉ đổi quan hệ thành so sánh số học và nhân thêm giá trị `x`.

In [37]:
def sum_subarray_minimums(arr):
    # Bài 907 (không lấy modulo): đại diện = vị trí min bên phải nhất
    n = len(arr)
    left = [-1] * n
    stack = []
    for i, x in enumerate(arr):
        while stack and arr[stack[-1]] >= x:     # trái: cho phép bằng -> pop cả phần tử bằng
            stack.pop()
        if stack:
            left[i] = stack[-1]
        stack.append(i)
    stack = []
    total = 0
    for i in range(n - 1, -1, -1):
        x = arr[i]
        while stack and arr[stack[-1]] > x:      # phải: cấm bằng -> dừng ở phần tử bằng
            stack.pop()
        right = stack[-1] if stack else n
        stack.append(i)
        total += x * (i - left[i]) * (right - i)
    return total


def sum_subarray_minimums_brute(arr):
    return sum(min(arr[l:r + 1]) for l in range(len(arr)) for r in range(l, len(arr)))


print(sum_subarray_minimums([3, 1, 2, 4]))     # ví dụ của bài 907: 17
random.seed(5)
for _ in range(200):
    arr = [random.randint(1, 5) for _ in range(random.randint(1, 8))]
    assert sum_subarray_minimums(arr) == sum_subarray_minimums_brute(arr)
print("Khung đóng góp + stack cho bài 907: PASS")

17
Khung đóng góp + stack cho bài 907: PASS


# 15. Bài tập vận dụng

Làm theo thứ tự, từ dễ đến khó. **Đừng xem đáp án** (mục 16) trước khi tự thử. Gợi ý 3 mức nằm ngay sau đề bài tập.

## Bài 1 — Đoán output (không chạy code)

Hãy đoán kết quả của từng dòng:

```python
print(12 & 10)
print(7 & 5 & 3)
print((13 & 4) == 4)
print((10 & 4) == 4)
print(bin(6 & 3))
print((5 & 2) == 2, 5 >= 2)
```

Và trả lời: mảng `[12, 13, 4]` có consistent không? Mảng `[6, 5]` thì sao?

## Bài 2 — Tính bằng tay

Với `nums = [3, 1, 2, 1]`:

1. Liệt kê cả 10 mảng con, đánh dấu mảng nào consistent.
2. Điền bảng của Cách 4 rồi cộng đóng góp, so với kết quả câu 1:

| i | x | `left[i]` | `right` | `(i - left) * (right - i)` |
|---:|---:|---:|---:|---:|
| 0 | 3 | ? | ? | ? |
| 1 | 1 | ? | ? | ? |
| 2 | 2 | ? | ? | ? |
| 3 | 1 | ? | ? | ? |

## Bài 3 — Điền chỗ trống: lượt trái → phải của Cách 4

Hoàn thiện hàm trả về mảng `left` (vật cản gần nhất bên trái, tức chỉ số `j < i` gần nhất mà `nums[j]` không chứa mọi bit của `nums[i]`; không có thì `-1`).

Kết quả mong đợi: `exercise_compute_left([5, 7, 2, 3, 2, 6]) == [-1, 0, 0, 2, 0, 4]`.

In [38]:
def exercise_compute_left(nums: List[int]) -> List[int]:
    n = len(nums)
    left = [-1] * n
    stack = []
    # TODO:
    # for i, x in enumerate(nums):
    #     while stack and ???:          # đỉnh stack chứa mọi bit của x
    #         stack.pop()
    #     if ???:
    #         left[i] = ???
    #     stack.append(???)
    # return left
    raise NotImplementedError("Hãy tự hoàn thiện hàm này trước khi xem đáp án.")


# Bỏ comment để test sau khi làm xong:
# assert exercise_compute_left([5, 7, 2, 3, 2, 6]) == [-1, 0, 0, 2, 0, 4]
# assert exercise_compute_left([1, 1, 1]) == [-1, -1, -1]

## Bài 4 — Tự viết lại Cách 3

Không nhìn lại mục 7, hãy tự viết hàm đếm số mảng con consistent bằng cách duy trì các nhóm `[a, lo, hi]` và bảng `last`.

Yêu cầu: chạy `O(n · 20)`, cho kết quả khớp brute force.

In [39]:
def exercise_count_and_groups(nums: List[int]) -> int:
    # TODO: duy trì groups = [[a, lo, hi], ...] và last = {giá trị: vị trí cuối}
    # Với mỗi r:
    #   1) cập nhật last[nums[r]]
    #   2) AND từng nhóm với nums[r], gộp nhóm kề trùng giá trị, thêm nhóm [nums[r], r, r]
    #   3) cộng số l tốt của từng nhóm
    raise NotImplementedError


# assert exercise_count_and_groups([5, 3, 1, 2]) == 6
# assert exercise_count_and_groups([1, 1, 1]) == 6
# assert exercise_count_and_groups([1, 3, 5, 1, 2]) == 10

## Bài 5 — Biến thể: OR bằng max

Một mảng con được gọi là **OR-consistent** nếu **OR bit** (`|`) của mọi phần tử bằng **max** của mảng con. Hãy đếm số mảng con OR-consistent.

Ví dụ:

```text
[5, 3, 1, 2] -> 6
[1, 2, 3]    -> 5     ([1], [2], [3], [2, 3], [1, 2, 3])
[4, 1, 5, 2] -> 6
```

Yêu cầu: viết cả bản brute force và bản `O(n)`, rồi đối chiếu trên dữ liệu ngẫu nhiên.

In [40]:
def exercise_count_or_equals_max(nums: List[int]) -> int:
    # TODO: bản O(n) "soi gương" Cách 4
    raise NotImplementedError


# assert exercise_count_or_equals_max([5, 3, 1, 2]) == 6
# assert exercise_count_or_equals_max([1, 2, 3]) == 5
# assert exercise_count_or_equals_max([4, 1, 5, 2]) == 6

## Bài 6 — Mini challenge: GCD bằng min

Cho mảng **số nguyên dương** `nums` (`nums[i] >= 1`). Đếm số mảng con có **ước chung lớn nhất (GCD)** của mọi phần tử bằng **min** của mảng con.

Ví dụ:

```text
[2, 4, 6, 3] -> 7
[6, 3, 9, 2] -> 7
[2, 4, 8]    -> 6
[3, 3, 3]    -> 6
```

Yêu cầu: lời giải `O(n)` (không tính chi phí phép `%`), dùng lại khung monotonic stack của Cách 4 với một quan hệ bắc cầu mới.

In [41]:
def exercise_count_gcd_equals_min(nums: List[int]) -> int:
    # TODO: tìm quan hệ thay cho "chứa mọi bit", rồi dùng lại khung Cách 4
    raise NotImplementedError


# assert exercise_count_gcd_equals_min([2, 4, 6, 3]) == 7
# assert exercise_count_gcd_equals_min([6, 3, 9, 2]) == 7
# assert exercise_count_gcd_equals_min([2, 4, 8]) == 6
# assert exercise_count_gcd_equals_min([3, 3, 3]) == 6

## Gợi ý (3 mức: hướng tư duy → cấu trúc / cú pháp → gần-code)

### Bài 1
1. Viết mỗi số thành 4 bit rồi AND theo cột.
2. `12 = 1100`, `10 = 1010`, `13 = 1101`, `4 = 0100`, `6 = 0110`, `5 = 0101`, `3 = 0011`, `2 = 0010`.
3. `1100 & 1010 = 1000`; với `[12, 13, 4]`: AND = `1100 & 1101 & 0100 = 0100 = 4` = min?

### Bài 2
1. Dùng điều kiện (B): "mọi phần tử chứa mọi bit của min".
2. `3 = 11`, `1 = 01`, `2 = 10`. Số 2 không chứa bit của 1, số 1 không chứa bit của 2 hay 3.
3. `i = 1` (x = 1): bên trái `3 = 11` chứa `1` → `left = -1`; bên phải `2 = 10` không chứa `1` → `right = 2`; đóng góp `(1 - (-1)) * (2 - 1) = 2`.

### Bài 3
1. Giống hệt `previous_smaller` ở mục 3.6, chỉ khác điều kiện pop.
2. Điều kiện pop: `(nums[stack[-1]] & x) == x`; nhớ kiểm tra `stack` khác rỗng trước.
3. `while stack and (nums[stack[-1]] & x) == x: stack.pop()`, rồi `if stack: left[i] = stack[-1]`, rồi `stack.append(i)`.

### Bài 4
1. Mỗi bước `r` có không quá 20 nhóm; mảng con `[l..r]` có AND = `a` tốt ⇔ `last[a] >= l`.
2. Danh sách mới `new_groups`; khi giá trị mới bằng `new_groups[-1][0]` thì chỉ nối `hi`.
3. Số `l` tốt của nhóm: `p = last.get(a, -1)`; `if p >= lo: total += min(hi, p) - lo + 1`.

### Bài 5
1. OR `>=` max (OR chỉ thêm bit). OR == max ⇔ max **chứa mọi bit** của mọi phần tử ⇔ mọi phần tử là submask của max.
2. Quan hệ mới: `nums[j]` là submask của `x` ⇔ `(nums[j] | x) == x`. Đại diện = vị trí max bên phải nhất.
3. Lấy `count_consistent_stack`, thay `(nums[top] & x) == x` bằng `(nums[top] | x) == x` ở cả hai lượt (giữ nguyên `nums[top] != x` ở lượt 2). Brute force: thay `&` bằng `|`, `min` bằng `max`.

### Bài 6
1. GCD `<=` min (GCD chia hết min). GCD == min ⇔ min **chia hết** mọi phần tử.
2. Quan hệ mới: "`x` chia hết `nums[j]`" ⇔ `nums[j] % x == 0`. Quan hệ chia hết có tính bắc cầu, và với số dương thì `x | y` kéo theo `y >= x` (giống F1).
3. Thay `(nums[top] & x) == x` bằng `nums[top] % x == 0` ở cả hai lượt. Brute force dùng `math.gcd`.

# 16. Đáp án tham khảo

Chỉ xem sau khi đã tự làm. Mỗi đáp án đều có `assert` để tự kiểm tra.

## Đáp án Bài 1

In [42]:
print(12 & 10)               # 1100 & 1010 = 1000 = 8
print(7 & 5 & 3)             # 111 & 101 & 011 = 001 = 1
print((13 & 4) == 4)         # 1101 có bit 2 -> True
print((10 & 4) == 4)         # 1010 không có bit 2 -> False
print(bin(6 & 3))            # 110 & 011 = 010 -> '0b10'
print((5 & 2) == 2, 5 >= 2)  # False True: lớn hơn nhưng không chứa bit

assert 12 & 10 == 8 and 7 & 5 & 3 == 1
assert and_of([12, 13, 4]) == min([12, 13, 4]) == 4     # [12, 13, 4] consistent
assert and_of([6, 5]) == 4 and min([6, 5]) == 5          # [6, 5] KHÔNG consistent
print("[12, 13, 4] consistent:", and_of([12, 13, 4]) == min([12, 13, 4]))
print("[6, 5] consistent:", and_of([6, 5]) == min([6, 5]))

8
1
True
False
0b10
False True
[12, 13, 4] consistent: True
[6, 5] consistent: False


## Đáp án Bài 2

Mảng con consistent của `[3, 1, 2, 1]`: `[3]`, `[1]`, `[2]`, `[1]` và `[3, 1]` → **5**.

| i | x | `left[i]` | `right` | `(i - left) * (right - i)` |
|---:|---:|---:|---:|---:|
| 0 | 3 | -1 | 1 | 1 |
| 1 | 1 | -1 | 2 | 2 |
| 2 | 2 | 1 | 3 | 1 |
| 3 | 1 | 2 | 4 | 1 |

Tổng `1 + 2 + 1 + 1 = 5`. Cell dưới kiểm tra lại bằng code:

In [43]:
show_table([3, 1, 2, 1])
print()
total = count_consistent_expand([3, 1, 2, 1], verbose=True)
assert total == count_consistent_bruteforce([3, 1, 2, 1]) == 5

l=0 r=0 sub=[3]            AND=3  min=3  Có
l=0 r=1 sub=[3, 1]         AND=1  min=1  Có
l=0 r=2 sub=[3, 1, 2]      AND=0  min=1  Không
l=0 r=3 sub=[3, 1, 2, 1]   AND=0  min=1  Không
l=1 r=1 sub=[1]            AND=1  min=1  Có
l=1 r=2 sub=[1, 2]         AND=0  min=1  Không
l=1 r=3 sub=[1, 2, 1]      AND=0  min=1  Không
l=2 r=2 sub=[2]            AND=2  min=2  Có
l=2 r=3 sub=[2, 1]         AND=0  min=1  Không
l=3 r=3 sub=[1]            AND=1  min=1  Có
Số mảng con consistent: 5

i=0, x=3: L=-1, R=1, đóng góp = (0-(-1)) * (1-0) = 1
i=1, x=1: L=-1, R=2, đóng góp = (1-(-1)) * (2-1) = 2
i=2, x=2: L=1, R=3, đóng góp = (2-(1)) * (3-2) = 1
i=3, x=1: L=2, R=4, đóng góp = (3-(2)) * (4-3) = 1


## Đáp án Bài 3

In [44]:
def answer_compute_left(nums: List[int]) -> List[int]:
    n = len(nums)
    left = [-1] * n
    stack = []
    for i, x in enumerate(nums):
        while stack and (nums[stack[-1]] & x) == x:
            stack.pop()
        if stack:
            left[i] = stack[-1]
        stack.append(i)
    return left


assert answer_compute_left([5, 7, 2, 3, 2, 6]) == [-1, 0, 0, 2, 0, 4]
assert answer_compute_left([1, 1, 1]) == [-1, -1, -1]
print(answer_compute_left([5, 7, 2, 3, 2, 6]))

[-1, 0, 0, 2, 0, 4]


## Đáp án Bài 4

In [45]:
def answer_count_and_groups(nums: List[int]) -> int:
    groups = []
    last = {}
    total = 0
    for r, x in enumerate(nums):
        last[x] = r
        new_groups = []
        for a, lo, hi in groups:
            a &= x
            if new_groups and new_groups[-1][0] == a:
                new_groups[-1][2] = hi
            else:
                new_groups.append([a, lo, hi])
        if new_groups and new_groups[-1][0] == x:
            new_groups[-1][2] = r
        else:
            new_groups.append([x, r, r])
        groups = new_groups
        for a, lo, hi in groups:
            p = last.get(a, -1)
            if p >= lo:
                total += min(hi, p) - lo + 1
    return total


assert answer_count_and_groups([5, 3, 1, 2]) == 6
assert answer_count_and_groups([1, 1, 1]) == 6
assert answer_count_and_groups([1, 3, 5, 1, 2]) == 10
random.seed(11)
for _ in range(200):
    arr = [random.randint(0, 7) for _ in range(random.randint(1, 9))]
    assert answer_count_and_groups(arr) == count_consistent_bruteforce(arr)
print("Đáp án Bài 4 PASS")

Đáp án Bài 4 PASS


## Đáp án Bài 5

OR chỉ **thêm** bit nên OR `>=` max. OR == max ⇔ max chứa mọi bit của mọi phần tử. Đại diện là vị trí max bên phải nhất; vật cản là phần tử **không phải submask** của `x` (bên phải: thêm cả phần tử bằng `x`).

In [46]:
def answer_or_brute(nums: List[int]) -> int:
    n = len(nums)
    count = 0
    for l in range(n):
        or_value, max_value = 0, nums[l]
        for r in range(l, n):
            or_value |= nums[r]
            max_value = max(max_value, nums[r])
            count += (or_value == max_value)
    return count


def answer_count_or_equals_max(nums: List[int]) -> int:
    n = len(nums)
    left = [-1] * n
    stack = []
    for i, x in enumerate(nums):
        while stack and (nums[stack[-1]] | x) == x:          # đỉnh là submask của x
            stack.pop()
        if stack:
            left[i] = stack[-1]
        stack.append(i)
    stack = []
    total = 0
    for i in range(n - 1, -1, -1):
        x = nums[i]
        while stack and nums[stack[-1]] != x and (nums[stack[-1]] | x) == x:
            stack.pop()
        right = stack[-1] if stack else n
        stack.append(i)
        total += (i - left[i]) * (right - i)
    return total


assert answer_count_or_equals_max([5, 3, 1, 2]) == 6
assert answer_count_or_equals_max([1, 2, 3]) == 5
assert answer_count_or_equals_max([4, 1, 5, 2]) == 6
random.seed(12)
for _ in range(300):
    arr = [random.randint(0, random.choice([3, 7, 100])) for _ in range(random.randint(1, 9))]
    assert answer_count_or_equals_max(arr) == answer_or_brute(arr)
print("Đáp án Bài 5 PASS")

Đáp án Bài 5 PASS


## Đáp án Bài 6

GCD chia hết mọi phần tử nên GCD `<=` min. GCD == min ⇔ min chia hết mọi phần tử. Quan hệ "`x` chia hết `y`" bắc cầu và (với số dương) kéo theo `y >= x`, nên khung Cách 4 dùng lại nguyên vẹn.

In [47]:
from math import gcd


def answer_gcd_brute(nums: List[int]) -> int:
    n = len(nums)
    count = 0
    for l in range(n):
        g, m = 0, nums[l]
        for r in range(l, n):
            g = gcd(g, nums[r])                 # gcd(0, v) = v
            m = min(m, nums[r])
            count += (g == m)
    return count


def answer_count_gcd_equals_min(nums: List[int]) -> int:
    n = len(nums)
    left = [-1] * n
    stack = []
    for i, x in enumerate(nums):
        while stack and nums[stack[-1]] % x == 0:          # x chia hết đỉnh
            stack.pop()
        if stack:
            left[i] = stack[-1]
        stack.append(i)
    stack = []
    total = 0
    for i in range(n - 1, -1, -1):
        x = nums[i]
        while stack and nums[stack[-1]] != x and nums[stack[-1]] % x == 0:
            stack.pop()
        right = stack[-1] if stack else n
        stack.append(i)
        total += (i - left[i]) * (right - i)
    return total


assert answer_count_gcd_equals_min([2, 4, 6, 3]) == 7
assert answer_count_gcd_equals_min([6, 3, 9, 2]) == 7
assert answer_count_gcd_equals_min([2, 4, 8]) == 6
assert answer_count_gcd_equals_min([3, 3, 3]) == 6
random.seed(13)
for _ in range(300):
    arr = [random.choice([1, 2, 3, 4, 6, 8, 12, 24, random.randint(1, 50)]) for _ in range(random.randint(1, 9))]
    assert answer_count_gcd_equals_min(arr) == answer_gcd_brute(arr)
print("Đáp án Bài 6 PASS")

Đáp án Bài 6 PASS


# 17. Tóm tắt — template ngắn dễ nhớ

## Ba câu cần nhớ

1. **AND `<=` min**, và **AND == min ⇔ mọi phần tử chứa mọi bit của min** ⇔ giá trị AND có mặt trong mảng con.
2. **Đóng góp**: mỗi mảng con consistent thuộc về vị trí min **bên phải nhất** → `ans = Σ (i - L) * (R - i)`.
3. **Monotonic stack dùng được với mọi quan hệ bắc cầu**: ở đây quan hệ là `(a & x) == x`; bên trái pop cả phần tử bằng, bên phải dừng ở phần tử bằng.

## Template

```python
class Solution:
    def countConsistent(self, nums: List[int]) -> int:
        n = len(nums)
        left = [-1] * n
        stack = []
        for i, x in enumerate(nums):                       # trái -> phải
            while stack and (nums[stack[-1]] & x) == x:    # đỉnh chứa x -> loại
                stack.pop()
            if stack:
                left[i] = stack[-1]
            stack.append(i)

        stack, ans = [], 0
        for i in range(n - 1, -1, -1):                     # phải -> trái
            x = nums[i]
            while stack and nums[stack[-1]] != x and (nums[stack[-1]] & x) == x:
                stack.pop()                                # chứa x và KHÁC x -> loại
            right = stack[-1] if stack else n
            stack.append(i)
            ans += (i - left[i]) * (right - i)
        return ans
```

## Template "log trick" (Cách 3) cho các bài AND/OR/GCD khác

```python
groups = []                                  # [giá_trị, lo, hi]
for r, x in enumerate(nums):
    new_groups = []
    for v, lo, hi in groups:
        v = v & x                            # hoặc v | x, gcd(v, x)
        if new_groups and new_groups[-1][0] == v:
            new_groups[-1][2] = hi
        else:
            new_groups.append([v, lo, hi])
    # thêm nhóm [x, r, r] (gộp nếu trùng giá trị), rồi xử lý các nhóm
```

# 18. Checklist trước khi submit

- [ ] Mình có viết điều kiện "chứa mọi bit" **có ngoặc**: `(nums[j] & x) == x`?
- [ ] Mình có nhầm `nums[j] >= x` với "chứa mọi bit" không? (`5 >= 2` nhưng `5` không chứa `2`.)
- [ ] Quy tắc phá hòa có **bất đối xứng** chưa: bên trái pop cả phần tử bằng `x`, bên phải dừng ở phần tử bằng `x`?
- [ ] Giá trị mặc định: `left[i] = -1` khi stack rỗng, `right = n` khi stack rỗng?
- [ ] Đóng góp là `(i - left[i]) * (right - i)` (không cộng/trừ 1 thừa)?
- [ ] Không có `break` sớm trong các bản `O(n²)` dùng để kiểm thử?
- [ ] Đã test `[1, 1, 1]` (trùng), `[2, 1, 0]` (số 0), `[2, 1]` (lớn hơn nhưng không chứa bit)?
- [ ] Đã đối chiếu với brute force trên vài trăm mảng nhỏ ngẫu nhiên?
- [ ] Nếu dịch sang Java/C++: dùng `long` cho đáp án và cho phép nhân?
- [ ] Độ phức tạp `O(n)` thời gian, `O(n)` bộ nhớ — phù hợp `n <= 2·10^5`.